# Hybrid Strategy: XGBoost Scoring + Regime-Aware Portfolio

Stocks scored by an **XGBoost model** trained on 76 features. Scores are EWM-smoothed for position stickiness, then applied with regime-aware active/held portfolio construction.

**Sector weighting modes** (toggle `SECTOR_WEIGHTING`):
- `none` — equal weight across all sectors; every sector long top-Q% + short bottom-Q%
- `fixed` — per-regime sector weights from training window: for each sector, the XGBoost-selected top-Q% vs bottom-Q% L/S alpha is computed, z-scored across sectors, and normalised to dollar-neutral weights. Positive weight → long that sector; negative → short it.
- `optimised` — same SLSQP optimisation as rule-based (L2-regularised Sharpe maximisation) but using XGBoost scores as the within-sector ranking signal instead of β·X.

## 1. Imports

In [ ]:
import copy
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import xgboost as xgb
from scipy.optimize import minimize
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from dateutil.relativedelta import relativedelta
from tqdm import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})

## 2. Configuration

In [ ]:
BASE_DIR     = Path('/Users/gabriel/Desktop/Sentiment Model')

RETURN_HORIZON  = '21d'
TARGET_COL      = f'target_ret_{RETURN_HORIZON}'
HORIZON_DAYS    = int(RETURN_HORIZON.replace('d', ''))
ANN_FACTOR      = 252 / HORIZON_DAYS   # 12 — IC metrics only
PORT_ANN_FACTOR = 252                  # portfolio P&L is 1d

DATA_START  = '2018-01-01'
DATA_END    = '2025-12-31'
TRAIN_YEARS = 2
TEST_YEARS  = 1.0

MCAP_FILTER_B   = 5.0
SECTOR_FILTER   = ['Basic Materials', 'Communication Services', 'Energy', 'Healthcare', 'Real Estate', 'Technology']
USE_DIRECTION   = True
SMOOTH_WINDOW   = 15
SLOPE_THRESHOLD = 0

ACTIVE_REGIMES = {
    'Extreme Fear', 'Fear-Falling', 'Fear-Rising',
    'Neutral',
    'Extreme Greed', 'Greed-Rising', 'Greed-Falling',
}

QUANTILE_GRID  = [0.05]   # top/bottom X% per sector to sweep
FIXED_LAM_GRID = [0.05]  # EWM speed values to sweep

# ── Sector weighting toggle ───────────────────────────────────────────────────
# 'none'      : equal weight, every sector long+short (current default)
# 'fixed'     : per-regime weights from training window using XGBoost factor performance
# 'optimised' : per-regime SLSQP weights (same method as rule_based_strategy)
# 'ic'        : IC-weighted tail alpha; long/short eligibility per regime
# 'sharpe'     : OOS z-scored per-leg Sharpe; long/short sized independently
# 'is_sharpe'  : IS recency-weighted per-leg Sharpe, z-scored; no OOS component
# 'is_alpha'   : IS recency-weighted per-leg excess alpha vs cross-sectional mean; no OOS component
SECTOR_WEIGHTING = 'fixed'

# Used by 'fixed' and 'optimised' modes
MIN_REGIME_DAYS = 30     # min training dates in regime before falling back to equal weight

# Used by 'optimised' mode only
L2_LAMBDA  = 0.5         # L2 penalty on w (prevents sector weight overfitting)
SOFT_EPS   = 0.05        # tanh smoothing of sign(w_k) — range [e^-4, e^-0.5]
FIXED_TAU  = 0.10        # softmax temperature (fixed; not learned)
N_RESTARTS = 3           # SLSQP random restarts
IC_DECAY            = 0.7    # per-year IC decay factor (weight = IC_DECAY ^ years_ago)
IC_MIN_THRESHOLD    = 0.01   # sectors below weighted IC excluded from weighting
ALPHA_MIN_THRESHOLD = 0.0    # min |alpha| for long/short eligibility

# ── Portfolio exposure ────────────────────────────────────────────────────────
# Only affects how evaluated returns are scaled. No effect on stock selection,
# sector weights, or any training logic.
#
# GROSS_EXPOSURE : total capital deployed (1.0 = 100% gross, 2.0 = 200% etc.)
# NET_EXPOSURE   : directional tilt in [-1, 1]
#   long_alloc  = GROSS_EXPOSURE * (1 + NET_EXPOSURE) / 2
#   short_alloc = GROSS_EXPOSURE * (1 - NET_EXPOSURE) / 2
#
# Examples:
#   GROSS=1.0, NET=0.0  → 0.5× long + 0.5× short  (default, dollar-neutral 1× gross)
#   GROSS=2.0, NET=0.0  → 1.0× long + 1.0× short  (200% gross, still neutral)
#   GROSS=3.0, NET=1/3  → 2.0× long + 1.0× short  (200%L / 100%S book)
#   GROSS=1.0, NET=1.0  → 1.0× long + 0.0× short  (long-only)
EXPOSURE_GRID  = [(1.5, 0.5), (2.0, 0.5), (1.5, 0.0), (3.0, 0.0), (1.0, 1.0), (2.0, 1.0)]  # (gross, net) pairs to sweep
GROSS_EXPOSURE = EXPOSURE_GRID[0][0]   # set per combo; init from grid
NET_EXPOSURE   = EXPOSURE_GRID[0][1]

# ── Markowitz L/S allocation ─────────────────────────────────────────────────
# USE_MARKOWITZ=True  → optimal long/short allocations estimated from training
#                       data using recency-weighted 2-asset Markowitz.
#                       GROSS_EXPOSURE still controls total scale; Markowitz
#                       decides the long/short split.
# USE_MARKOWITZ=False → fixed split from GROSS_EXPOSURE and NET_EXPOSURE above.
# MARKOWITZ_CLIP      → max ratio of short_alloc / long_alloc (prevents extreme
#                       short-heavy solutions from noisy estimates).
MARKOWITZ_GRID = [False]               # L/S allocation method to sweep
MARKOWITZ_CLIP  = 1.0   # short_alloc <= long_alloc * MARKOWITZ_CLIP

# ── Exposure brake ───────────────────────────────────────────────────────────
# USE_EXPOSURE_BRAKE=True  → scale long allocation by FNG/VIX brake signal
#                            during fear/volatility events. Scale: 0.20–1.0.
# USE_EXPOSURE_BRAKE=False → no adjustment, full long allocation always applied.
USE_EXPOSURE_BRAKE = True

# ── XGBoost model architecture ───────────────────────────────────────────────
# USE_THREE_MODEL=False → single XGBRegressor (current behaviour)
# USE_THREE_MODEL=True  → mixture: pos-regressor + neg-regressor + binary classifier
#   prediction = P(y>0)*E[y|y>0] + (1-P(y>0))*E[y|y<=0]
USE_THREE_MODEL = False

XGB_REG_PARAMS = dict(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=50,
    gamma=1.0, reg_alpha=0.1, reg_lambda=1.0,
    objective='reg:squarederror', tree_method='hist',
    n_jobs=-1, random_state=42, verbosity=0,
)
# XGB_REG_PARAMS = dict(
#     n_estimators=500, max_depth=4, learning_rate=0.05,
#     subsample=0.8, colsample_bytree=0.6, colsample_bylevel=0.7, min_child_weight=8,
#     gamma=0.2, reg_alpha=0.1, reg_lambda=1.0,
#     objective='reg:squarederror', eval_metric='rmse',
#     tree_method='hist', n_jobs=-1, random_state=42, verbosity=0,
# )
XGB_CLS_PARAMS = dict(
    n_estimators=2500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.6, min_child_weight=10,
    gamma=0.5, reg_alpha=0.1, reg_lambda=1.0,
    objective='binary:logistic', eval_metric='logloss',
    tree_method='hist', n_jobs=-1, random_state=42, verbosity=0,
)
XGB_PARAMS = XGB_REG_PARAMS   # alias for single-model path

# ── Early stopping ────────────────────────────────────────────────────────────
# EARLY_STOPPING_ROUNDS > 0 → hold out the last ES_VAL_FRAC of training dates
# as a validation set; stop when eval metric does not improve for N rounds.
# Set EARLY_STOPPING_ROUNDS = 0 to disable.
EARLY_STOPPING_ROUNDS = 0
ES_VAL_FRAC           = 0.15   # fraction of training dates held out for early stopping

# ── Per-regime XGBoost toggle ─────────────────────────────────────────────────
# SHARED_XGB=True  → one model trained on all training data (default)
# SHARED_XGB=False → separate model per active regime; regimes with < MIN_REGIME_DAYS
#                    fall back to the global model
SHARED_XGB = True

HALF_LIFE_YEARS = 2.0

SNAPSHOT_DIR = BASE_DIR / 'portfolio_snapshots'

print(f'Target           : {TARGET_COL}')
print(f'WF folds         : {TRAIN_YEARS}yr expanding train -> {TEST_YEARS}yr test')
print(f'MCAP filter      : >= ${MCAP_FILTER_B}B')
print(f'SECTOR_FILTER    : {SECTOR_FILTER}')
print(f'QUANTILE_GRID    : {QUANTILE_GRID}')
print(f'FIXED_LAM_GRID   : {FIXED_LAM_GRID}')
print(f'Active regimes   : {sorted(ACTIVE_REGIMES)}')
print(f'SECTOR_WEIGHTING : {SECTOR_WEIGHTING}')
print(f'EXPOSURE_GRID    : {EXPOSURE_GRID}')
print(f'MARKOWITZ_GRID   : {MARKOWITZ_GRID}  (CLIP={MARKOWITZ_CLIP})')
print(f'USE_THREE_MODEL  : {USE_THREE_MODEL}')
print(f'SHARED_XGB       : {SHARED_XGB}')
print(f'USE_EXPOSURE_BRAKE: {USE_EXPOSURE_BRAKE}')
print(f'Snapshots        : {SNAPSHOT_DIR}')
print(f'IC_DECAY           : {IC_DECAY}')
print(f'IC_MIN_THRESHOLD   : {IC_MIN_THRESHOLD}')
print(f'ALPHA_MIN_THRESHOLD: {ALPHA_MIN_THRESHOLD}')
if SECTOR_WEIGHTING == 'optimised':
    print(f'  L2_LAMBDA={L2_LAMBDA}  SOFT_EPS={SOFT_EPS}  FIXED_TAU={FIXED_TAU}  N_RESTARTS={N_RESTARTS}')
elif SECTOR_WEIGHTING == 'ic':
    print(f'  IC_DECAY={IC_DECAY}  IC_MIN_THRESHOLD={IC_MIN_THRESHOLD}  ALPHA_MIN_THRESHOLD={ALPHA_MIN_THRESHOLD}')

## 3. Regime Map

Same definitions as `rule_based_strategy.ipynb`. **Active regimes** trigger basket rebalance; **inactive** hold the last basket and track its actual 1d returns.

In [ ]:
REGIME_BINS  = [0, 25, 45, 55, 75, 100]
REGIME_ORDER = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
REGIME_COLORS = {
    'Extreme Fear':  '#d62728',
    'Fear':          '#ff7f0e',
    'Neutral':       '#aec7e8',
    'Greed':         '#2ca02c',
    'Extreme Greed': '#17becf',
}
all_possible = {
    'Extreme Fear', 'Fear-Rising', 'Fear-Falling', 'Fear-Stable',
    'Neutral', 'Greed-Rising', 'Greed-Falling', 'Greed-Stable', 'Extreme Greed',
}
print('Active regimes:')
for r in sorted(ACTIVE_REGIMES): print(f'  {r}')
print('\nInactive (held):')
for r in sorted(all_possible - ACTIVE_REGIMES): print(f'  {r}')

## 4. Data Loading

In [ ]:
STOCK_COLS = [
    'date', 'ticker',
    # Momentum
    'roc_5', 'roc_10', 'roc_21', 'roc_63',
    'macd_hist_fast_pct', 'macd_hist_std_pct',
    'rsi_5', 'rsi_21', 'rsi_63',
    # Trend / EMA
    'ema5_above_ema21', 'ema10_above_ema21', 'ema21_above_ema63', 'ema21_above_ema126',
    'dist_to_ema_10_pct', 'dist_to_ema_21_pct', 'dist_to_ema_63_pct', 'dist_to_ema_126_pct',
    # Volatility
    'vol_std_21', 'vol_std_63', 'vol_std_zscore_21', 'vol_std_zscore_63',
    'parkinson_vol_21', 'atr_21_pct_close', 'return_skew_21',
    # Bollinger
    'bb_21_pct_from_upper', 'bb_21_pct_from_lower',
    # Volume
    'vol_zscore_21', 'vol_zscore_63', 'vol_ratio_21', 'vol_ratio_5',
    'vol_price_alignment_21', 'obv_dist_to_ema_21_pct', 'mfm_avg_21',
    # Stochastics
    'stoch_k_21', 'stoch_k_63', 'stoch_k_d_diff_21',
    # Candlestick
    'gap_pct', 'body_size_pct', 'upper_wick_pct', 'lower_wick_pct', 'weekly_body_size_pct',
    # Factor betas
    'beta_spy', 'beta_tlt', 'beta_^vix', 'beta_sphb', 'beta_splv',
    'beta_mtum', 'beta_qual', 'beta_iwd', 'beta_iwf',
    'beta_xlk', 'beta_xle', 'beta_xlf', 'beta_xly',
    # Calendar
    'dow_1', 'dow_2', 'dow_3', 'dow_4',
    'first_week_of_month', 'last_week_of_month',
    # Targets
    'target_ret_1d', 'target_ret_3d', 'target_ret_5d',
    'target_ret_10d', 'target_ret_21d',
]
batches = sorted((BASE_DIR / 'stock_training_data_final').glob('batch_*.parquet'))
print(f'Loading {len(batches)} parquet batches...')
stock_df = pd.concat(
    [pd.read_parquet(p, columns=STOCK_COLS) for p in tqdm(batches)],
    ignore_index=True
)
if stock_df['date'].dt.tz is not None:
    stock_df['date'] = stock_df['date'].dt.tz_localize(None)
stock_df['date'] = pd.to_datetime(stock_df['date']).dt.normalize()
print(f'Stock data: {stock_df.shape}')

In [ ]:
MKT_COLS = [
    'vix', 'vix_zscore_21', 'egarch_vol',
    'spx_dist_to_ma200', 'spx_mom_3m', 'spx_drawdown',
    'yield_10y_3m', 'yield_30y_10y',
    'credit_spread_mom_1m', 'hyg_tlt_ratio_mom_1m',
    'rsp_spy_spread_1m', 'cyclical_defensive_spread_1m',
    'fear_greed',
    'fng_q2_fear', 'fng_q3_neutral', 'fng_q4_greed', 'fng_q5_extreme_greed',
    'fng_diff_5',
]
mkt_path = BASE_DIR / 'market_training_data_final' / 'market_state_vector.parquet'
mkt_df   = pd.read_parquet(mkt_path, columns=MKT_COLS).reset_index()
mkt_df   = mkt_df.rename(columns={mkt_df.columns[0]: 'date'})
if mkt_df['date'].dt.tz is not None:
    mkt_df['date'] = mkt_df['date'].dt.tz_localize(None)
mkt_df['date'] = pd.to_datetime(mkt_df['date']).dt.normalize()
mkt_df = mkt_df.sort_values('date').reset_index(drop=True)
print(f'Market data: {mkt_df.shape}')

In [ ]:
def rolling_fng_slope(fng_series, window):
    slopes = np.full(len(fng_series), np.nan)
    vals = fng_series.values
    x    = np.arange(window, dtype=float)
    for i in range(window, len(vals)):
        y = vals[i - window: i]
        if not np.isnan(y).any():
            slopes[i] = np.polyfit(x, y, 1)[0]
    return pd.Series(slopes, index=fng_series.index)

def slope_to_direction(slope_series, threshold):
    d = pd.Series('Stable', index=slope_series.index)
    d[slope_series >=  threshold] = 'Rising'
    d[slope_series < -threshold]  = 'Falling'
    return d

mkt_df['regime_base'] = pd.cut(
    mkt_df['fear_greed'], bins=REGIME_BINS, labels=REGIME_ORDER, include_lowest=True
).astype(str)
mkt_df['fng_slope']  = rolling_fng_slope(mkt_df['fear_greed'], SMOOTH_WINDOW)
mkt_df['direction']  = slope_to_direction(mkt_df['fng_slope'], SLOPE_THRESHOLD)

def build_regime_label(row):
    base = row['regime_base']
    if not USE_DIRECTION or base in ('Extreme Fear', 'Extreme Greed', 'Neutral'):
        return base
    return f"{base}-{row['direction']}"

mkt_df['regime']      = mkt_df.apply(build_regime_label, axis=1)
fng_regime_series     = mkt_df.set_index('date')['regime_base']
print('Regime distribution:')
print(mkt_df['regime'].value_counts().sort_index().rename('days'))

# ── Exposure brake signal ─────────────────────────────────────────────────────
_vix_ma20   = mkt_df['vix'].rolling(20).mean()
_fng_ma20   = mkt_df['fear_greed'].rolling(20).mean()
_in_fear      = mkt_df['fear_greed'] < 45
_fng_below_ma = mkt_df['fear_greed'] < _fng_ma20 * 0.80
_vix_spike    = (mkt_df['vix'] > 25) & (mkt_df['vix'] > _vix_ma20 * 1.25)
_fng_scale = np.where(
    (mkt_df['fng_diff_5'] < -20) & _in_fear & _fng_below_ma,
    (1.0 + (mkt_df['fng_diff_5'] + 20) / 10).clip(lower=0.20), 1.0)
_vix_scale = np.where(
    _vix_spike,
    (1.0 - (mkt_df['vix'] - 25) / 20).clip(lower=0.20), 1.0)
_raw_scale = pd.DataFrame({'f': _fng_scale, 'v': _vix_scale}).min(axis=1)
def _asym_ewm(raw, span=3):
    alpha = 2 / (span + 1)
    s = np.ones(len(raw))
    for i in range(1, len(raw)):
        s[i] = min(raw.iloc[i], (1-alpha)*s[i-1] + alpha*raw.iloc[i])
    return pd.Series(s, index=raw.index)
_smoothed = _asym_ewm(_raw_scale, span=3)
_brake    = np.where(_smoothed > 0.89, 1.0, _smoothed)
# 1-day lag: signal from day t applied to position on day t+1
mkt_df['brake_scale'] = pd.Series(_brake, index=mkt_df.index).shift(1).fillna(1.0)
brake_scale_map = mkt_df.set_index('date')['brake_scale'].to_dict()
print(f'Brake signal computed: {(mkt_df["brake_scale"] < 1.0).sum()} brake days in dataset')

In [ ]:
meta = (
    pd.read_csv(BASE_DIR / 'us_stocks_500m.csv',
                usecols=['ticker', 'sector', 'market_cap'])
    .drop_duplicates('ticker')
)
meta['market_cap_b'] = meta['market_cap'] / 1e9

EMB_COLS = (
    ['ticker']
    + [f'sector_emb_{i}'   for i in range(6)]
    + [f'industry_emb_{i}' for i in range(16)]
    + [f'country_emb_{i}'  for i in range(8)]
    + ['mcap_small', 'mcap_mid', 'mcap_large', 'mcap_mega']
)
emb_df = pd.read_csv(BASE_DIR / 'stock_market_features.csv', usecols=EMB_COLS)

df = (
    stock_df
    .merge(mkt_df[['date'] + MKT_COLS + ['regime_base', 'regime']], on='date', how='left')
    .merge(meta[['ticker', 'sector', 'market_cap_b']], on='ticker', how='left')
    .merge(emb_df, on='ticker', how='left')
)
df = df[(df['date'] >= DATA_START) & (df['date'] <= DATA_END)].copy()
df['month_sin'] = np.sin(2 * np.pi * df['date'].dt.month / 12).astype(np.float32)
df['month_cos'] = np.cos(2 * np.pi * df['date'].dt.month / 12).astype(np.float32)

if MCAP_FILTER_B is not None:
    before = df['ticker'].nunique()
    df = df[df['market_cap_b'] >= MCAP_FILTER_B].copy()
    print(f'MCAP filter >= ${MCAP_FILTER_B}B: {before:,} -> {df["ticker"].nunique():,} tickers')
if SECTOR_FILTER is not None:
    print(f'Sector filter active (portfolio only): {SECTOR_FILTER}')

core_required = ['roc_21', 'vol_std_21', 'beta_spy', 'fear_greed',
                 TARGET_COL, 'target_ret_1d', 'sector', 'regime']
df.dropna(subset=core_required, inplace=True)
df.reset_index(drop=True, inplace=True)

ALL_SECTORS = sorted(df['sector'].dropna().unique())
print(f'Merged   : {df.shape}  |  Tickers: {df["ticker"].nunique():,}  |  Dates: {df["date"].nunique():,}')
print(f'Sectors  ({len(ALL_SECTORS)}): {ALL_SECTORS}')

## 5. Feature Definitions

In [ ]:
STOCK_FEATURES = [
    'roc_5', 'roc_21', 'roc_63',
    'vol_std_21', 'vol_std_63', 'vol_std_zscore_21',
    'atr_21_pct_close',
    'rsi_21', 'rsi_63',
    'ema5_above_ema21', 'ema21_above_ema63',
    'dist_to_ema_21_pct', 'dist_to_ema_63_pct',
    'vol_zscore_21', 'vol_ratio_21',
    'beta_spy', 'beta_tlt', 'beta_^vix', 'beta_sphb', 'beta_splv',
    'stoch_k_21',
    'dow_1', 'dow_2', 'dow_3', 'dow_4',
]
MARKET_FEATURES = list(MKT_COLS)
EMBED_FEATURES  = (
    [f'sector_emb_{i}'   for i in range(6)]
    + [f'industry_emb_{i}' for i in range(16)]
    + [f'country_emb_{i}'  for i in range(8)]
    + ['mcap_small', 'mcap_mid', 'mcap_large', 'mcap_mega']
)
FEATURE_COLS   = [c for c in STOCK_FEATURES + MARKET_FEATURES + EMBED_FEATURES if c in df.columns]
NON_STOCK_COLS = [c for c in MARKET_FEATURES + EMBED_FEATURES if c in df.columns]

print(f'Total features : {len(FEATURE_COLS)}')
print(f'  Stock        : {len([c for c in STOCK_FEATURES if c in df.columns])}')
print(f'  Market       : {len([c for c in MARKET_FEATURES if c in df.columns])}')
print(f'  Embeddings   : {len([c for c in EMBED_FEATURES if c in df.columns])}')

## 6. Walk-Forward Folds

In [ ]:
def generate_wf_folds(start_year, end_year, min_train_years, test_years, step_years=1):
    folds = []
    test_months = round(test_years * 12)
    step_months = round(step_years * 12)
    test_start  = pd.Timestamp(f'{start_year + min_train_years}-01-01')
    while True:
        train_start = pd.Timestamp(f'{start_year}-01-01')
        train_end   = test_start - pd.Timedelta(days=1)
        test_end    = test_start + relativedelta(months=test_months) - pd.Timedelta(days=1)
        if test_end > pd.Timestamp(f'{end_year}-12-31'):
            break
        te_s_str = test_start.strftime('%Y-%m')
        label = f'Train {train_start.year}-{train_end.year} | Test {te_s_str}'
        folds.append((label, train_start, train_end, test_start, test_end))
        test_start += relativedelta(months=step_months)
    return folds

WF_FOLDS = generate_wf_folds(2018, 2025, TRAIN_YEARS, TEST_YEARS, step_years=TEST_YEARS)
print('Walk-forward folds:')
for name, tr_s, tr_e, te_s, te_e in WF_FOLDS:
    print(f'  {name}  [{(te_s - tr_s).days // 365}yr train]')

## 7. Strategy Functions

**Preprocessing**: CS z-score stock features per date; StandardScaler on market+embeddings (fit on train). XGBoost is tree-based so doesn't strictly need scaling, but this keeps the pipeline consistent with the baseline notebook.

**Sector weight modes** (`SECTOR_WEIGHTING`):
- `none`: `build_date_portfolio` uses equal weight, longs top-Q% AND shorts bottom-Q% in every sector simultaneously.
- `fixed`: `compute_fixed_sector_weights` uses XGBoost training predictions to score each sector's L/S alpha (top-Q% return minus bottom-Q% return), z-scores across sectors, and normalises to dollar-neutral w_k. Positive w_k → long that sector; negative → short it.
- `optimised`: `compute_optimised_sector_weights` runs SLSQP (same as rule-based Stage 2) with L2 regularisation to maximise training 1d portfolio Sharpe. XGBoost scores are the within-sector ranking signal.

In [ ]:
def make_recency_weights(dates, half_life_years=HALF_LIFE_YEARS):
    """Exponential decay. Set HALF_LIFE_YEARS=None for uniform weights."""
    if half_life_years is None:
        return np.ones(len(dates), dtype=np.float32)
    d = pd.to_datetime(dates)
    days_ago = np.asarray((d.max() - d) / pd.Timedelta('1D'), dtype=np.float32)
    return np.exp(-np.log(2) * days_ago / (half_life_years * 365.25))


def preprocess_fold(train, test):
    """CS z-score stock features per date; StandardScaler on market+embeddings."""
    train, test = train.copy(), test.copy()
    for col in STOCK_FEATURES:
        if col not in train.columns:
            continue
        train[col] = train.groupby('date')[col].transform(
            lambda x: (x - x.mean()) / (x.std() + 1e-8))
        test[col]  = test.groupby('date')[col].transform(
            lambda x: (x - x.mean()) / (x.std() + 1e-8))
    scaler = StandardScaler()
    train[NON_STOCK_COLS] = scaler.fit_transform(train[NON_STOCK_COLS].fillna(0))
    test[NON_STOCK_COLS]  = scaler.transform(test[NON_STOCK_COLS].fillna(0))
    X_train  = train[FEATURE_COLS].fillna(0).values.astype(np.float32)
    X_test   = test[FEATURE_COLS].fillna(0).values.astype(np.float32)
    y_train  = train[TARGET_COL].values.astype(np.float32)
    weights  = make_recency_weights(train['date'])
    return X_train, y_train, X_test, train, test, weights


def _es_fit(model, X_tr, y_tr, w_tr, X_val=None, y_val=None,
            X_full=None, y_full=None, w_full=None):
    """
    Fit with optional two-pass early stopping:
      Pass 1 — train on X_tr (85%), early-stop on X_val (15%) to find best n_estimators.
      Pass 2 — retrain on X_full (100%) using that n_estimators, no early stopping.
    Falls back to plain fit if early stopping is disabled or no val set provided.
    """
    if EARLY_STOPPING_ROUNDS > 0 and X_val is not None:
        model.set_params(early_stopping_rounds=EARLY_STOPPING_ROUNDS)
        model.fit(X_tr, y_tr, sample_weight=w_tr,
                  eval_set=[(X_tr, y_tr), (X_val, y_val)],
                  verbose=False)
        model._es_result = model.evals_result()
        model._es_best   = model.best_iteration
        # Deepcopy pass-1 model before overwriting with pass-2 retrain
        model._pass1_model = copy.deepcopy(model)
        if X_full is not None:
            n_best = max(100, model.best_iteration + 1)
            model.set_params(n_estimators=n_best, early_stopping_rounds=None)
            _p2_eval = [(X_full, y_full)]
            if X_val is not None:
                _p2_eval.append((X_val, y_val))
            model.fit(X_full, y_full, sample_weight=w_full, eval_set=_p2_eval, verbose=False)
            model._pass2_result = model.evals_result()
            # Compute pass-2 train/val metrics manually for _xgb_summary compatibility
            is_cls = isinstance(model, xgb.XGBClassifier)
            if is_cls:
                eps = 1e-7
                p_val   = model.predict_proba(X_val)[:, 1].clip(eps, 1 - eps)
                p_full  = model.predict_proba(X_full)[:, 1].clip(eps, 1 - eps)
                model._pass2_val_metric   = float(-np.mean(
                    y_val  * np.log(p_val)   + (1 - y_val)  * np.log(1 - p_val)))
                model._pass2_train_metric = float(-np.mean(
                    y_full * np.log(p_full)  + (1 - y_full) * np.log(1 - p_full)))
            else:
                model._pass2_val_metric   = float(np.sqrt(np.mean((y_val  - model.predict(X_val))  ** 2)))
                model._pass2_train_metric = float(np.sqrt(np.mean((y_full - model.predict(X_full)) ** 2)))
            model._pass2_n_trees = n_best
    else:
        _eval = [(X_tr, y_tr)]
        if X_val is not None:
            _eval.append((X_val, y_val))
        model.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=_eval, verbose=False)
    return model


def _xgb_summary(model, label):
    """Print train/val loss for pass-1 and (if two-pass) pass-2 side by side."""
    try:
        res = getattr(model, '_es_result', None) or model.evals_result()
    except Exception:
        return
    if not res:
        return
    keys    = list(res.keys())
    metric  = list(res[keys[0]].keys())[0]
    tr_hist = res[keys[0]][metric]
    vl_hist = res[keys[1]][metric] if len(keys) > 1 else None
    best    = getattr(model, '_es_best', getattr(model, 'best_iteration', len(tr_hist) - 1))
    n1      = len(tr_hist)
    tr1     = tr_hist[best]
    vl1     = vl_hist[best] if vl_hist else None
    r1      = f'{vl1/tr1:.3f}' if vl1 and tr1 > 1e-10 else 'n/a'
    has_p2  = hasattr(model, '_pass2_val_metric')
    if has_p2:
        tr2 = model._pass2_train_metric
        vl2 = model._pass2_val_metric
        n2  = model._pass2_n_trees
        r2  = f'{vl2/tr2:.3f}' if tr2 > 1e-10 else 'n/a'
        print(f'    {label:18s}  '
              f'pass1: n={n1:3d} best={best:3d} {metric}(tr/vl)={tr1:.5f}/{vl1:.5f} ratio={r1}  |  '
              f'pass2: n={n2:3d} {metric}(tr/vl)={tr2:.5f}/{vl2:.5f} ratio={r2}')
    elif vl1 is not None:
        print(f'    {label:18s}  trees={n1:3d}  best={best:3d}  '
              f'{metric}(tr/vl)={tr1:.5f}/{vl1:.5f}  ratio={r1}')
    else:
        print(f'    {label:18s}  trees={n1:3d}  train_{metric}={tr_hist[-1]:.5f}')


def _plot_learning_curve(global_model, fold_name, lc_step=20):
    """Plot train/val loss every lc_step trees for each sub-model in the fold."""
    import matplotlib.pyplot as plt

    def _extract(model):
        use_p2 = EARLY_STOPPING_ROUNDS > 0 and hasattr(model, '_pass2_result')
        try:
            res = (model._pass2_result if use_p2
                   else getattr(model, '_es_result', None) or model.evals_result())
        except Exception:
            return None
        if not res:
            return None
        keys   = list(res.keys())
        metric = list(res[keys[0]].keys())[0]
        tr_h   = res[keys[0]][metric]
        vl_h   = res[keys[1]][metric] if len(keys) > 1 else None
        n      = len(tr_h)
        idx    = list(range(0, n, lc_step))
        if n - 1 not in idx:
            idx.append(n - 1)
        trees  = [i + 1 for i in idx]
        return dict(trees=trees,
                    train=[tr_h[i] for i in idx],
                    val=[vl_h[i] for i in idx] if vl_h else None,
                    metric=metric, pass2=use_p2)

    submodels = ([(global_model['pos'], 'pos_reg'),
                  (global_model['neg'], 'neg_reg'),
                  (global_model['cls'], 'classifier')]
                 if isinstance(global_model, dict)
                 else [(global_model, 'global_reg')])

    ncols = len(submodels)
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 4), squeeze=False)
    fig.suptitle(f'Learning Curves — {fold_name}', fontsize=12)
    for ax, (model, label) in zip(axes[0], submodels):
        d = _extract(model)
        if d is None:
            ax.set_title(f'{label} — no data')
            continue
        suffix = ' (pass 2)' if d['pass2'] else ''
        ax.plot(d['trees'], d['train'], 'b-o', markersize=3, label='train')
        if d['val'] is not None:
            ax.plot(d['trees'], d['val'], 'r-o', markersize=3, label='val')
        ax.set_title(f'{label}{suffix}')
        ax.set_xlabel('n_estimators')
        ax.set_ylabel(d['metric'])
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def train_xgboost(X_train, y_train, weights, X_val=None, y_val=None,
                  X_full=None, y_full=None, w_full=None):
    model = xgb.XGBRegressor(**XGB_PARAMS)
    _es_fit(model, X_train, y_train, weights, X_val, y_val, X_full, y_full, w_full)
    _xgb_summary(model, 'global_reg')
    return model


def train_xgboost_mixture(X_train, y_train, weights, X_val=None, y_val=None,
                          X_full=None, y_full=None, w_full=None):
    """Train pos-regressor, neg-regressor, and binary classifier. Returns dict."""
    pos_mask = y_train > 0
    neg_mask = ~pos_mask
    y_cls    = pos_mask.astype(np.int32)
    pos_model = xgb.XGBRegressor(**XGB_REG_PARAMS)
    neg_model = xgb.XGBRegressor(**XGB_REG_PARAMS)
    cls_model = xgb.XGBClassifier(**XGB_CLS_PARAMS)
    # For pos/neg regressors, filter val set to matching sign rows
    val_pos = (X_val[y_val > 0], y_val[y_val > 0]) if X_val is not None and (y_val > 0).any() else (None, None)
    val_neg = (X_val[y_val <= 0], y_val[y_val <= 0]) if X_val is not None and (y_val <= 0).any() else (None, None)
    # Two-pass: for pos/neg regressors, filter X_full by sign of y_full
    if X_full is not None and y_full is not None and w_full is not None:
        full_pos_mask = y_full > 0
        full_neg_mask = ~full_pos_mask
        full_pos = (X_full[full_pos_mask], y_full[full_pos_mask], w_full[full_pos_mask])
        full_neg = (X_full[full_neg_mask], y_full[full_neg_mask], w_full[full_neg_mask])
        y_full_cls = (y_full > 0).astype(np.int32)
    else:
        full_pos = full_neg = (None, None, None)
        y_full_cls = None
    _es_fit(pos_model, X_train[pos_mask], y_train[pos_mask], weights[pos_mask], *val_pos, *full_pos)
    _xgb_summary(pos_model, 'pos_reg')
    _es_fit(neg_model, X_train[neg_mask], y_train[neg_mask], weights[neg_mask], *val_neg, *full_neg)
    _xgb_summary(neg_model, 'neg_reg')
    y_cls_val = (y_val > 0).astype(np.int32) if y_val is not None else None
    _es_fit(cls_model, X_train, y_cls, weights, X_val, y_cls_val, X_full, y_full_cls, w_full)
    _xgb_summary(cls_model, 'classifier')
    return {'pos': pos_model, 'neg': neg_model, 'cls': cls_model}


def predict_xgboost_mixture(models, X):
    """Produce mixture prediction: p_pos*mu_pos + (1-p_pos)*mu_neg."""
    mu_pos = models['pos'].predict(X).astype(np.float64)
    mu_neg = models['neg'].predict(X).astype(np.float64)
    p_pos  = models['cls'].predict_proba(X)[:, 1].astype(np.float64)
    return (p_pos * mu_pos + (1.0 - p_pos) * mu_neg).astype(np.float64)


def xgb_predict(model_or_mixture, X):
    """Unified predict: handles single model or three-model mixture dict."""
    if isinstance(model_or_mixture, dict):
        return predict_xgboost_mixture(model_or_mixture, X)
    return model_or_mixture.predict(X).astype(np.float64)


def xgb_feature_importances(model_or_mixture):
    """Return feature importances from single model or classifier in mixture."""
    if isinstance(model_or_mixture, dict):
        return model_or_mixture['cls'].feature_importances_
    return model_or_mixture.feature_importances_


def preprocess_train_only(train):
    """CS z-score stock features per date; StandardScaler on market+embeddings (train only)."""
    train = train.copy()
    for col in STOCK_FEATURES:
        if col not in train.columns:
            continue
        train[col] = train.groupby('date')[col].transform(
            lambda x: (x - x.mean()) / (x.std() + 1e-8))
    scaler = StandardScaler()
    train[NON_STOCK_COLS] = scaler.fit_transform(train[NON_STOCK_COLS].fillna(0))
    X = train[FEATURE_COLS].fillna(0).values.astype(np.float32)
    y = train[TARGET_COL].values.astype(np.float32)
    w = make_recency_weights(train['date'])
    return X, y, train, w


def _default_allocs():
    return (GROSS_EXPOSURE * (1 + NET_EXPOSURE) / 2,
            GROSS_EXPOSURE * (1 - NET_EXPOSURE) / 2)

def _net_exposure_pnl(l, s_pnl, long_alloc=None, short_alloc=None):
    """
    Compute portfolio P&L. long_alloc/short_alloc override global GROSS/NET when provided.
    """
    if long_alloc is None or short_alloc is None:
        long_alloc, short_alloc = _default_allocs()
    return long_alloc * l + short_alloc * s_pnl


def compute_markowitz_alloc(train_df, regime_sw):
    """
    Find the optimal long fraction x of GROSS_EXPOSURE via 1-D Sharpe maximisation.

    Uses per-sector selection and regime sector weights to match the actual
    portfolio construction in build_date_portfolio exactly.

    MARKOWITZ_CLIP controls the max short-to-long ratio:
      short_alloc <= long_alloc * MARKOWITZ_CLIP
      => (1-x) <= x * CLIP  =>  x >= 1 / (1 + CLIP)
    With CLIP=1.0: x in [0.5, 1.0]  (net-neutral to fully long)
    With CLIP=2.0: x in [0.33, 1.0] (up to 2x more short than long)
    """
    if any(isinstance(sw, tuple) for sw in regime_sw.values()):
        _la, _sa = _default_allocs()
        return _la, _sa, float('nan'), float('nan')
    from scipy.optimize import minimize_scalar
    daily_l, daily_s, dates = [], [], []
    for date, ddf in train_df.groupby('date'):
        ddf = ddf.dropna(subset=['xgb_score', 'target_ret_1d'])
        if len(ddf) < 10:
            continue
        regime = ddf['regime'].iloc[0]
        sw     = regime_sw.get(regime, None)
        long_contrib, short_contrib = 0.0, 0.0
        long_w_sum,   short_w_sum   = 0.0, 0.0
        for sector, grp in ddf.groupby('sector'):
            grp = grp.dropna(subset=['xgb_score', 'target_ret_1d'])
            n   = len(grp)
            if n < 5:
                continue
            n_pick = max(1, int(np.ceil(n * QUANTILE)))
            ranked = grp.sort_values('xgb_score')
            if sw is None:
                long_contrib  += ranked.iloc[-n_pick:]['target_ret_1d'].mean()
                long_w_sum    += 1.0
                short_contrib += ranked.iloc[:n_pick]['target_ret_1d'].mean()
                short_w_sum   += 1.0
            else:
                w_k = sw.get(sector, 0.0)
                if w_k > 1e-6:
                    long_contrib += w_k * ranked.iloc[-n_pick:]['target_ret_1d'].mean()
                    long_w_sum   += w_k
                elif w_k < -1e-6:
                    short_contrib += w_k * ranked.iloc[:n_pick]['target_ret_1d'].mean()
                    short_w_sum   += abs(w_k)
        if long_w_sum == 0 or short_w_sum == 0:
            continue
        l = long_contrib / long_w_sum
        s = -(short_contrib / short_w_sum) if sw is None else (short_contrib / short_w_sum)
        daily_l.append(l)
        daily_s.append(s)
        dates.append(date)
    if len(daily_l) < 20:
        _la, _sa = _default_allocs(); return _la, _sa, float('nan'), float('nan')
    l_arr = np.array(daily_l, dtype=np.float64)
    s_arr = np.array(daily_s, dtype=np.float64)
    w = make_recency_weights(pd.Series(dates)).astype(np.float64)
    w = w / w.sum()

    def neg_sharpe(x):
        port = x * l_arr + (1.0 - x) * s_arr
        mu_p  = (w * port).sum()
        var_p = (w * (port - mu_p) ** 2).sum()
        return -mu_p / np.sqrt(var_p + 1e-12)

    x_lo = 1.0 / (1.0 + MARKOWITZ_CLIP)   # max short bound
    res  = minimize_scalar(neg_sharpe, bounds=(x_lo, 1.0), method='bounded')
    x    = float(res.x)
    la   = x * GROSS_EXPOSURE
    sa   = (1.0 - x) * GROSS_EXPOSURE
    mu_l = (w * l_arr).sum()
    mu_s = (w * s_arr).sum()
    return la, sa, mu_l, mu_s

In [ ]:
def compute_fixed_sector_weights(train_regime_df):
    """
    Per-sector L/S alpha using XGBoost scores as the ranking signal.
    For each sector: top-QUANTILE stocks (by xgb_score) mean 21d return
    minus bottom-QUANTILE mean 21d return = sector alpha.
    Z-score alpha across sectors → normalise to dollar-neutral weights.
    Positive w_k → long that sector's top stocks; negative → short its bottom stocks.
    Falls back to empty dict (equal-weight) if regime has too few data.
    """
    sector_alpha = {}
    for sector, grp in train_regime_df.groupby('sector'):
        grp = grp.dropna(subset=['xgb_score', TARGET_COL])
        n   = len(grp)
        if n < max(5, int(np.ceil(n * QUANTILE)) * 2):
            continue
        n_pick = max(1, int(np.ceil(n * QUANTILE)))
        top = grp.nlargest(n_pick,  'xgb_score')
        bot = grp.nsmallest(n_pick, 'xgb_score')
        sector_alpha[sector] = top[TARGET_COL].mean() - bot[TARGET_COL].mean()

    if len(sector_alpha) < 2:
        return {}

    alpha_s = pd.Series(sector_alpha)
    z = (alpha_s - alpha_s.mean()) / (alpha_s.std() + 1e-8)

    # Dollar-neutral normalisation: Σ(w_k>0) = 1, Σ(w_k<0) = -1
    w = z.copy()
    pos_sum = w[w > 0].sum()
    neg_sum = w[w < 0].abs().sum()
    if pos_sum > 1e-8: w[w > 0] /= pos_sum
    if neg_sum > 1e-8: w[w < 0] /= neg_sum

    return w.to_dict()


def compute_optimised_sector_weights(train_regime_df):
    """
    SLSQP optimisation of sector weights w_k to maximise 1d portfolio Sharpe
    on the training window, using XGBoost scores as the within-sector ranking signal.
    Identical to rule_based_strategy Stage 2 except xgb_score replaces β·X.
    Returns dollar-neutral w_k dict. Falls back to fixed weights if too little data.
    """
    sub = train_regime_df.dropna(subset=['xgb_score', 'target_ret_1d', 'sector']).copy()
    n_dates = sub['date'].nunique()
    sectors = sorted(sub['sector'].dropna().unique())
    K       = len(sectors)
    sec_idx = {s: i for i, s in enumerate(sectors)}

    if n_dates < MIN_REGIME_DAYS or K < 2:
        return compute_fixed_sector_weights(train_regime_df)

    # Build integer ticker index for vectorised EWM
    all_tickers   = sorted(sub['ticker'].unique())
    ticker_to_idx = {t: i for i, t in enumerate(all_tickers)}
    N_T           = len(all_tickers)
    one_minus_lam = 1.0 - FIXED_LAM

    # Pre-group into {sec_idx: (tidx, raw_score, R_1d)} per date
    date_data = []
    for _, dgrp in sub.groupby('date'):
        d = {}
        for s, sgrp in dgrp.groupby('sector'):
            tidx = np.array([ticker_to_idx[t] for t in sgrp['ticker'].values])
            raw  = sgrp['xgb_score'].values.astype(np.float64)
            R_1d = sgrp['target_ret_1d'].values.astype(np.float64)
            d[sec_idx[s]] = (tidx, raw, R_1d)
        date_data.append(d)

    def portfolio_returns(w):
        ema_arr     = np.zeros(N_T)
        initialized = np.zeros(N_T, dtype=bool)
        r_series    = []
        for d in date_data:
            rp = 0.0
            for k, (tidx, raw, R_1d) in d.items():
                if len(R_1d) < 2:
                    continue
                prev          = np.where(initialized[tidx], ema_arr[tidx], raw)
                ema           = one_minus_lam * prev + FIXED_LAM * raw
                ema_arr[tidx] = ema
                initialized[tidx] = True
                dir_k  = np.tanh(w[k] / SOFT_EPS)
                logits = ema * dir_k / FIXED_TAU
                logits -= logits.max()
                alpha  = np.exp(logits) / np.exp(logits).sum()
                rp    += w[k] * (alpha @ R_1d)
            r_series.append(rp)
        return np.array(r_series)

    def loss(w):
        r   = portfolio_returns(w)
        std = r.std()
        if std < 1e-10:
            return 0.0
        return -(r.mean() / std * np.sqrt(252)) + L2_LAMBDA * np.sum(w ** 2)

    constraints = [{'type': 'eq', 'fun': lambda w: w.sum()}]
    rng = np.random.default_rng(42)
    best_res, best_val = None, np.inf
    for _ in range(N_RESTARTS):
        w0  = rng.standard_normal(K) * 0.1
        w0 -= w0.mean()
        res = minimize(loss, w0, method='SLSQP', constraints=constraints,
                       options={'maxiter': 500, 'ftol': 1e-7})
        if res.fun < best_val:
            best_val, best_res = res.fun, res

    if best_res is None:
        return compute_fixed_sector_weights(train_regime_df)

    w_opt   = best_res.x
    pos_sum = w_opt[w_opt > 0].sum()
    if pos_sum > 1e-8:
        w_opt = w_opt / pos_sum   # dollar-neutral normalisation

    return dict(zip(sectors, w_opt))




def compute_fold_sector_ic(test_df, quantile):
    """Within-sector OOS IC using pooled tail observations across all OOS dates."""
    from scipy.stats import spearmanr
    result = {}
    for sector, grp in test_df.groupby('sector'):
        rows = []
        for _, day in grp.groupby('date'):
            k = max(1, int(len(day) * quantile))
            rows.append(pd.concat([day.nlargest(k, 'xgb_pred'),
                                   day.nsmallest(k, 'xgb_pred')]))
        if not rows:
            continue
        pooled = pd.concat(rows)
        if len(pooled) < 10:
            continue
        rho, _ = spearmanr(pooled['xgb_pred'], pooled[TARGET_COL])
        result[sector] = float(rho)
    return result


def get_weighted_sector_ic(ic_history, fold_idx, quantile, current_date):
    """Time-decay-weighted IC from prior folds only. Returns {sector: weighted_ic}."""
    prior_keys = [(i, quantile) for i in range(fold_idx) if (i, quantile) in ic_history]
    if not prior_keys:
        return {}
    current_date = pd.to_datetime(current_date)
    weighted = {}
    total_w  = {}
    for key in prior_keys:
        entry     = ic_history[key]
        years_ago = (current_date - pd.to_datetime(entry['end_date'])).days / 365.25
        w = IC_DECAY ** years_ago
        for sector, ic in entry['ic'].items():
            weighted[sector] = weighted.get(sector, 0.0) + w * ic
            total_w[sector]  = total_w.get(sector, 0.0) + w
    return {s: weighted[s] / total_w[s]
            for s in weighted
            if weighted[s] / total_w[s] >= IC_MIN_THRESHOLD}


def compute_ic_sector_weights(train_regime_df, weighted_ic, quantile):
    """
    Per-regime: top-Q%/bottom-Q% alpha x |IC| -> (long_sw, short_sw).
    Each dict normalized to sum 1.0. Returns (None, None) if no IC available.
    """
    if not weighted_ic:
        return None, None
    long_raw  = {}
    short_raw = {}
    for sector, grp in train_regime_df.groupby('sector'):
        ic = weighted_ic.get(sector)
        if ic is None:
            continue
        dates = sorted(grp['date'].unique())
        rw    = make_recency_weights(dates)
        long_rets, short_rets, wts = [], [], []
        for i, (date, day) in enumerate(grp.groupby('date')):
            k = max(1, int(len(day) * quantile))
            long_rets.append(day.nlargest(k, 'xgb_score')[TARGET_COL].mean())
            short_rets.append(day.nsmallest(k, 'xgb_score')[TARGET_COL].mean())
            wts.append(rw[i])
        if not wts:
            continue
        wts = np.array(wts) / sum(wts)
        la  = float(np.dot(wts, long_rets))
        sa  = float(np.dot(wts, short_rets))
        if la  >  ALPHA_MIN_THRESHOLD:
            long_raw[sector]  = la  * abs(ic)
        if sa < -ALPHA_MIN_THRESHOLD:
            short_raw[sector] = abs(sa) * abs(ic)

    def _norm(d):
        total = sum(d.values())
        return {s: v / total for s, v in d.items()} if total > 0 else None

    return _norm(long_raw), _norm(short_raw)


def compute_fold_sector_sharpes(test_df, quantile):
    """Per-sector OOS unweighted Sharpe and ann% for long/short legs across all OOS dates."""
    score_col = 'xgb_pred' if 'xgb_pred' in test_df.columns else 'xgb_score'
    result = {}
    for sector, grp in test_df.groupby('sector'):
        long_rets, short_rets = [], []
        for _, day in grp.groupby('date'):
            k = max(1, int(len(day) * quantile))
            long_rets.append(day.nlargest(k, score_col)['target_ret_1d'].mean())
            short_rets.append(day.nsmallest(k, score_col)['target_ret_1d'].mean())
        if len(long_rets) < 20:
            continue
        l = np.array(long_rets)
        s = np.array(short_rets)
        l_sh  = float(l.mean() / (l.std() + 1e-8) * np.sqrt(252)) if l.std() > 1e-8 else 0.0
        s_sh  = float(-s.mean() / (s.std() + 1e-8) * np.sqrt(252)) if s.std() > 1e-8 else 0.0
        l_ann = float(l.mean() * 252 * 100)
        s_ann = float(-s.mean() * 252 * 100)
        result[sector] = {'long': l_sh, 'short': s_sh, 'long_ann': l_ann, 'short_ann': s_ann}
    return result


def get_weighted_sector_sharpes(sharpe_history, fold_idx, quantile, current_date):
    """Decay-weighted per-leg Sharpe from prior folds only. Returns (long_dict, short_dict)."""
    prior_keys = [(i, quantile) for i in range(fold_idx) if (i, quantile) in sharpe_history]
    if not prior_keys:
        return {}, {}
    long_w, short_w, total_w = {}, {}, {}
    for key in prior_keys:
        entry = sharpe_history[key]
        years_ago = max(0.0, (pd.Timestamp(current_date) - pd.Timestamp(entry['end_date'])).days / 365.25)
        w = IC_DECAY ** years_ago
        for sector, sh in entry['sharpes'].items():
            long_w[sector]  = long_w.get(sector, 0.0)  + w * sh['long']
            short_w[sector] = short_w.get(sector, 0.0) + w * sh['short']
            total_w[sector] = total_w.get(sector, 0.0) + w
    wl = {s: long_w[s]  / total_w[s] for s in long_w}
    ws = {s: short_w[s] / total_w[s] for s in short_w}
    return wl, ws


def compute_sharpe_sector_weights(train_regime_df, wl_sharpes, ws_sharpes, quantile):
    """Per-regime in-sample alpha x z-scored OOS Sharpe -> (long_sw, short_sw) normalised to 1.0."""
    if not wl_sharpes and not ws_sharpes:
        return None, None

    def _zscores(d):
        if not d:
            return {}
        vals = np.array(list(d.values()))
        mu, sigma = vals.mean(), vals.std()
        if sigma < 1e-8:
            return {s: 0.0 for s in d}
        return {s: (v - mu) / sigma for s, v in d.items()}

    zl     = _zscores(wl_sharpes)
    zs_map = _zscores(ws_sharpes)

    score_col = 'xgb_pred' if 'xgb_pred' in train_regime_df.columns else 'xgb_score'
    long_alpha, short_alpha = {}, {}
    for sector, grp in train_regime_df.groupby('sector'):
        dates = sorted(grp['date'].unique())
        rw    = make_recency_weights(dates)
        long_rets, short_rets = [], []
        for _, day in grp.groupby('date'):
            k = max(1, int(len(day) * quantile))
            long_rets.append(day.nlargest(k, score_col)['target_ret_1d'].mean())
            short_rets.append(day.nsmallest(k, score_col)['target_ret_1d'].mean())
        if not long_rets:
            continue
        wts = np.array(rw) / sum(rw)
        long_alpha[sector]  = float(np.dot(wts, long_rets))
        short_alpha[sector] = float(np.dot(wts, short_rets))

    long_raw, short_raw = {}, {}
    for sector, z in zl.items():
        la = long_alpha.get(sector, 0.0)
        if z > 0 and la > ALPHA_MIN_THRESHOLD:
            long_raw[sector] = z * la
    for sector, z in zs_map.items():
        sa = short_alpha.get(sector, 0.0)
        if z > 0 and sa < -ALPHA_MIN_THRESHOLD:
            short_raw[sector] = z * abs(sa)

    def _norm(d):
        total = sum(d.values())
        return {s: v / total for s, v in d.items()} if total > 0 else None

    return _norm(long_raw), _norm(short_raw)

def compute_is_sharpe_sector_weights(train_regime_df, quantile):
    """
    Per-regime IS Sharpe: per-date top/bottom Q% by XGB score within each sector,
    21d forward returns (TARGET_COL), recency-weighted Sharpe (HALF_LIFE_YEARS=2).
    Long and short legs computed independently.
    Z-score across sectors -> keep positive -> normalise to 1.0.
    """
    score_col = 'xgb_pred' if 'xgb_pred' in train_regime_df.columns else 'xgb_score'
    long_sharpes, short_sharpes = {}, {}

    for sector, grp in train_regime_df.groupby('sector'):
        grp   = grp.dropna(subset=[score_col, TARGET_COL])
        dates = sorted(grp['date'].unique())
        if len(dates) < 20:
            continue
        rw  = make_recency_weights(dates)
        wts = np.array(rw) / sum(rw)

        long_rets, short_rets = [], []
        for _, day in grp.groupby('date'):
            k = max(1, int(np.ceil(len(day) * quantile)))
            long_rets.append(day.nlargest(k,  score_col)[TARGET_COL].mean())
            short_rets.append(day.nsmallest(k, score_col)[TARGET_COL].mean())

        def _recency_sharpe(rets, wts_arr):
            a = np.array(rets, dtype=np.float64)
            mu  = float(np.dot(wts_arr, a))
            std = float(np.sqrt(np.dot(wts_arr, (a - mu) ** 2)))
            return mu / std * np.sqrt(ANN_FACTOR) if std > 1e-8 else np.nan

        l_sh = _recency_sharpe(long_rets,  wts)
        s_sh = _recency_sharpe(short_rets, wts)
        if np.isfinite(l_sh):
            long_sharpes[sector]  = l_sh
        if np.isfinite(s_sh):
            short_sharpes[sector] = -s_sh  # negate: good short has negative 21d returns

    def _zscore_norm(d):
        if not d:
            return None
        vals = np.array(list(d.values()))
        mu, sigma = vals.mean(), vals.std()
        if sigma < 1e-8:
            return None
        zs  = {s: (v - mu) / sigma for s, v in d.items()}
        pos = {s: z for s, z in zs.items() if z > 0}
        if not pos:
            return None
        total = sum(pos.values())
        return {s: v / total for s, v in pos.items()}

    return _zscore_norm(long_sharpes), _zscore_norm(short_sharpes)

def compute_is_alpha_sector_weights(train_regime_df, quantile):
    """
    Per-regime IS alpha: per-date top/bottom Q% by XGB score within each sector,
    excess 21d return vs cross-sectional mean (all stocks, all sectors).
    Recency-weighted mean excess return per leg, independently.
    Z-score across sectors -> keep positive -> normalise to 1.0.
    """
    score_col = 'xgb_pred' if 'xgb_pred' in train_regime_df.columns else 'xgb_score'

    # Per-date benchmark: mean 21d return across ALL stocks in this regime window
    bench_by_date = (train_regime_df.dropna(subset=[TARGET_COL])
                     .groupby('date')[TARGET_COL].mean())

    long_alphas, short_alphas = {}, {}

    for sector, grp in train_regime_df.groupby('sector'):
        grp   = grp.dropna(subset=[score_col, TARGET_COL])
        dates = sorted(grp['date'].unique())
        if len(dates) < 20:
            continue
        rw  = make_recency_weights(dates)
        wts = np.array(rw) / sum(rw)

        long_excess, short_excess = [], []
        for _, day in grp.groupby('date'):
            k     = max(1, int(np.ceil(len(day) * quantile)))
            bench = bench_by_date.get(day['date'].iloc[0], np.nan)
            long_excess.append(day.nlargest(k,  score_col)[TARGET_COL].mean() - bench)
            short_excess.append(bench - day.nsmallest(k, score_col)[TARGET_COL].mean())

        l_alpha = float(np.dot(wts, long_excess))
        s_alpha = float(np.dot(wts, short_excess))
        if np.isfinite(l_alpha):
            long_alphas[sector]  = l_alpha
        if np.isfinite(s_alpha):
            short_alphas[sector] = s_alpha

    def _zscore_norm(d):
        if not d:
            return None
        vals = np.array(list(d.values()))
        mu, sigma = vals.mean(), vals.std()
        if sigma < 1e-8:
            return None
        zs  = {s: (v - mu) / sigma for s, v in d.items()}
        pos = {s: z for s, z in zs.items() if z > 0}
        if not pos:
            return None
        total = sum(pos.values())
        return {s: v / total for s, v in pos.items()}

    return _zscore_norm(long_alphas), _zscore_norm(short_alphas)

def get_sector_weights(train_regime_df, weighted_ic=None, weighted_long_sharpes=None, weighted_short_sharpes=None):
    """Dispatch to the appropriate weight function based on SECTOR_WEIGHTING."""
    if SECTOR_FILTER is not None:
        train_regime_df = train_regime_df[train_regime_df['sector'].isin(SECTOR_FILTER)]
    if SECTOR_WEIGHTING == 'ic':
        return compute_ic_sector_weights(train_regime_df, weighted_ic or {}, QUANTILE)
    elif SECTOR_WEIGHTING == 'sharpe':
        return compute_sharpe_sector_weights(train_regime_df, weighted_long_sharpes or {}, weighted_short_sharpes or {}, QUANTILE)
    elif SECTOR_WEIGHTING == 'is_sharpe':
        return compute_is_sharpe_sector_weights(train_regime_df, QUANTILE)
    elif SECTOR_WEIGHTING == 'is_alpha':
        return compute_is_alpha_sector_weights(train_regime_df, QUANTILE)
    elif SECTOR_WEIGHTING == 'fixed':
        return compute_fixed_sector_weights(train_regime_df)
    elif SECTOR_WEIGHTING == 'optimised':
        return compute_optimised_sector_weights(train_regime_df)
    return None   # 'none' mode


def eval_regime_train_sharpe(sub, sw):
    """
    Compute annualised Sharpe on training data using EWM XGBoost scores and
    sector weights sw (None = equal weight: every sector long top + short bottom).
    """
    if isinstance(sw, tuple):
        long_sw, short_sw = sw
        merged = dict(long_sw or {})
        for s, w in (short_sw or {}).items():
            merged[s] = merged.get(s, 0.0) - w
        sw = merged if merged else None
    sub = sub.dropna(subset=['xgb_score', 'target_ret_1d', 'sector']).copy()
    if len(sub) < 20:
        return np.nan
    ema_scores = {}
    r_series   = []
    for date, ddf in sub.groupby('date'):
        tickers    = ddf['ticker'].values
        raw_scores = ddf['xgb_score'].values.astype(np.float64)
        for t, r in zip(tickers, raw_scores):
            ema_scores[t] = (1.0 - FIXED_LAM) * ema_scores.get(t, r) + FIXED_LAM * r

        long_contrib, short_contrib = 0.0, 0.0
        long_w_sum, short_w_sum     = 0.0, 0.0
        for sector, grp in ddf.groupby('sector'):
            grp    = grp.dropna(subset=['xgb_score', 'target_ret_1d'])
            n      = len(grp)
            n_pick = max(1, int(np.ceil(n * QUANTILE)))
            if n < 5:
                continue
            ema_arr = np.array([ema_scores.get(t, float(s))
                                for t, s in zip(grp['ticker'], grp['xgb_score'])])
            ranked  = grp.assign(_ema=ema_arr).sort_values('_ema')
            if sw is None:
                top = ranked.iloc[-n_pick:]
                bot = ranked.iloc[:n_pick]
                long_contrib  += top['target_ret_1d'].mean()
                long_w_sum    += 1.0
                short_contrib += bot['target_ret_1d'].mean()
                short_w_sum   += 1.0
            else:
                w_k = sw.get(sector, 0.0)
                if w_k > 1e-6:
                    top = ranked.iloc[-n_pick:]
                    long_contrib += w_k * top['target_ret_1d'].mean()
                    long_w_sum   += w_k
                elif w_k < -1e-6:
                    bot = ranked.iloc[:n_pick]
                    short_contrib += w_k * bot['target_ret_1d'].mean()
                    short_w_sum   += abs(w_k)

        if long_w_sum > 0 or short_w_sum > 0:
            l = long_contrib / long_w_sum if long_w_sum > 0 else 0.0
            if sw is None:
                # short_contrib = Σ raw short returns (positive = bad for us)
                s_pnl = -(short_contrib / short_w_sum) if short_w_sum > 0 else 0.0
            else:
                # short_contrib = Σ(w_k<0)*r_k — already in P&L direction
                s_pnl = short_contrib / short_w_sum if short_w_sum > 0 else 0.0
            r_series.append(_net_exposure_pnl(l, s_pnl))

    if len(r_series) < 10:
        return np.nan
    r   = np.array(r_series)
    std = r.std()
    return r.mean() / std * np.sqrt(252) if std > 1e-10 else np.nan

In [ ]:
def compute_ic_metrics(y_true, y_pred, dates, tickers):
    ic_global, _ = spearmanr(y_pred, y_true)
    results_df   = pd.DataFrame({'date': dates, 'ticker': tickers,
                                 'y_true': y_true, 'y_pred': y_pred})
    daily_ic = (
        results_df.groupby('date')
        .apply(lambda g: spearmanr(g['y_pred'], g['y_true'])[0] if len(g) >= 5 else np.nan)
        .dropna()
    )
    mean_ic = daily_ic.mean()
    icir    = mean_ic / daily_ic.std() if daily_ic.std() > 0 else np.nan
    return {
        'IC (global)':   ic_global,
        'Mean Daily IC': mean_ic,
        'ICIR':          icir,
        'Hit Rate':      np.mean(np.sign(y_pred) == np.sign(y_true)),
        'R2':            r2_score(y_true, y_pred),
        'RMSE':          np.sqrt(mean_squared_error(y_true, y_pred)),
        '_daily_ic':     daily_ic,
    }


def compute_strategy_metrics(port_series):
    s = port_series.dropna()
    if len(s) < 5:
        return {k: np.nan for k in
                ['Ann. Return', 'Ann. Vol', 'Sharpe', 'Sortino', 'Calmar', 'Max DD', 'Win Rate', 'VaR 5%', 'CVaR 5%']}
    ann_ret  = s.mean() * PORT_ANN_FACTOR
    ann_vol  = s.std()  * np.sqrt(PORT_ANN_FACTOR)
    sharpe   = ann_ret / ann_vol if ann_vol > 0 else np.nan
    downside = s[s < 0].std() * np.sqrt(PORT_ANN_FACTOR)
    sortino  = ann_ret / downside if downside > 0 else np.nan
    cum      = (1 + s).cumprod()
    max_dd   = ((cum - cum.cummax()) / cum.cummax()).min()
    calmar   = ann_ret / abs(max_dd) if max_dd < 0 else np.nan
    var5     = float(np.percentile(s, 5))
    cvar5    = float(s[s <= var5].mean()) if (s <= var5).any() else np.nan
    return {'Ann. Return': ann_ret, 'Ann. Vol': ann_vol, 'Sharpe': sharpe,
            'Sortino': sortino, 'Calmar': calmar, 'Max DD': max_dd,
            'Win Rate': float((s > 0).mean()), 'VaR 5%': var5, 'CVaR 5%': cvar5}

In [ ]:
def build_date_portfolio(date_df, prev_scores, last_basket, is_active, sector_weights,
                         long_alloc=None, short_alloc=None):
    """
    Apply EWM-smoothed XGBoost scores to one date's cross-section.

    sector_weights: None → 'none' mode (equal weight, every sector long+short)
                    dict  → 'fixed'/'optimised' (w_k per sector; pos=long, neg=short)

    On ACTIVE days  : update EWM scores, re-select basket, compute 1d returns.
    On HELD days    : update EWM scores (keeps signal current), compute 1d returns
                      of the last active basket using today's prices.

    Returns: port_ret, long_ret, short_ret, score_records, new_scores, new_basket
    """
    date_df   = date_df.copy()
    tickers   = date_df['ticker'].values
    raw_preds = date_df['xgb_pred'].values

    # EWM update — runs every day regardless of active/held
    new_scores = {}
    ema_col    = np.empty(len(date_df))
    for i, (t, r) in enumerate(zip(tickers, raw_preds)):
        ema = (1.0 - FIXED_LAM) * prev_scores.get(t, float(r)) + FIXED_LAM * float(r)
        new_scores[t] = ema
        ema_col[i]    = ema
    date_df['_score'] = ema_col

    if isinstance(sector_weights, tuple) and sector_weights[0] is None and sector_weights[1] is None:
        sector_weights = None

    if is_active:
        long_tickers, short_tickers = set(), set()
        score_records = []

        if sector_weights is None:
            # ── 'none' mode: every sector long top-Q% AND short bottom-Q% ────
            for sector, grp in date_df.groupby('sector'):
                if SECTOR_FILTER is not None and sector not in SECTOR_FILTER:
                    continue
                grp    = grp.dropna(subset=['_score', 'target_ret_1d'])
                n      = len(grp)
                n_pick = max(1, int(np.ceil(n * QUANTILE)))
                if n < 5:
                    continue
                ranked = grp.sort_values('_score')
                top    = ranked.iloc[-n_pick:]
                bot    = ranked.iloc[:n_pick]
                long_tickers.update(top['ticker'].tolist())
                short_tickers.update(bot['ticker'].tolist())
                for _, row in top.iterrows():
                    score_records.append({'ticker': row['ticker'], 'y_pred': row['_score'],
                                          'y_true': row[TARGET_COL], 'side': 'long'})
                for _, row in bot.iterrows():
                    score_records.append({'ticker': row['ticker'], 'y_pred': row['_score'],
                                          'y_true': row[TARGET_COL], 'side': 'short'})
        elif isinstance(sector_weights, tuple):
            # ── 'ic' mode: independent long/short books ──────────────────────
            long_sw_b, short_sw_b = sector_weights
            for sector, grp in date_df.groupby('sector'):
                grp    = grp.dropna(subset=['_score', 'target_ret_1d'])
                n      = len(grp)
                n_pick = max(1, int(np.ceil(n * QUANTILE)))
                if n < 5:
                    continue
                ranked = grp.sort_values('_score')
                if sector in long_sw_b:
                    top  = ranked.iloc[-n_pick:]
                    long_tickers.update(top['ticker'].tolist())
                    for _, row in top.iterrows():
                        score_records.append({'ticker': row['ticker'], 'y_pred': row['_score'],
                                              'y_true': row[TARGET_COL], 'side': 'long',
                                              'sector_w': long_sw_b[sector]})
                if sector in short_sw_b:
                    bot  = ranked.iloc[:n_pick]
                    short_tickers.update(bot['ticker'].tolist())
                    for _, row in bot.iterrows():
                        score_records.append({'ticker': row['ticker'], 'y_pred': row['_score'],
                                              'y_true': row[TARGET_COL], 'side': 'short',
                                              'sector_w': -short_sw_b[sector]})
        else:
            # ── 'fixed'/'optimised' mode: w_k determines sector direction ────
            for sector, grp in date_df.groupby('sector'):
                w_k = sector_weights.get(sector, 0.0)
                if abs(w_k) < 1e-6:
                    continue
                grp    = grp.dropna(subset=['_score', 'target_ret_1d'])
                n      = len(grp)
                n_pick = max(1, int(np.ceil(n * QUANTILE)))
                if n < 5:
                    continue
                ranked = grp.sort_values('_score')
                if w_k > 0:          # long sector: top stocks
                    sel  = ranked.iloc[-n_pick:]
                    side = 'long'
                    long_tickers.update(sel['ticker'].tolist())
                else:                # short sector: bottom stocks
                    sel  = ranked.iloc[:n_pick]
                    side = 'short'
                    short_tickers.update(sel['ticker'].tolist())
                for _, row in sel.iterrows():
                    score_records.append({'ticker': row['ticker'], 'y_pred': row['_score'],
                                          'y_true': row[TARGET_COL], 'side': side,
                                          'sector_w': w_k})

        new_basket = {'long': long_tickers, 'short': short_tickers,
                      'sector_weights': sector_weights}
    else:
        new_basket    = last_basket
        score_records = []

    # ── Compute 1d returns for the basket ────────────────────────────────────
    basket = new_basket if new_basket else {'long': set(), 'short': set()}
    long_df  = date_df[date_df['ticker'].isin(basket.get('long',  set()))].dropna(subset=['target_ret_1d'])
    short_df = date_df[date_df['ticker'].isin(basket.get('short', set()))].dropna(subset=['target_ret_1d'])

    sw = basket.get('sector_weights', None)
    if isinstance(sw, tuple) and sw[0] is None and sw[1] is None:
        sw = None

    if sw is None:
        # Equal-weight: average long return and average short return.
        # s_pnl = -short_raw because raw short return is positive-when-bad (we are short).
        l_raw = float(long_df['target_ret_1d'].mean())  if len(long_df)  > 0 else 0.0
        s_raw = float(short_df['target_ret_1d'].mean()) if len(short_df) > 0 else 0.0
        port_ret  = _net_exposure_pnl(l_raw, -s_raw, long_alloc, short_alloc)
        long_ret  = l_raw if len(long_df)  > 0 else np.nan
        short_ret = s_raw if len(short_df) > 0 else np.nan
    elif isinstance(sw, tuple):
        # IC mode: two independent normalized books
        long_sw_r, short_sw_r = sw
        long_contrib, short_contrib = 0.0, 0.0
        long_w_sum,   short_w_sum   = 0.0, 0.0
        for sector, grp in long_df.groupby('sector'):
            w_k = (long_sw_r or {}).get(sector, 0.0)
            if w_k < 1e-6: continue
            long_contrib += w_k * grp['target_ret_1d'].mean()
            long_w_sum   += w_k
        for sector, grp in short_df.groupby('sector'):
            w_k = (short_sw_r or {}).get(sector, 0.0)
            if w_k < 1e-6: continue
            short_contrib += w_k * grp['target_ret_1d'].mean()
            short_w_sum   += w_k
        l     = long_contrib  / long_w_sum  if long_w_sum  > 0 else 0.0
        s_pnl = -(short_contrib / short_w_sum) if short_w_sum > 0 else 0.0
        port_ret  = _net_exposure_pnl(l, s_pnl, long_alloc, short_alloc)
        long_ret  = l     if long_w_sum  > 0 else np.nan
        short_ret = s_pnl if short_w_sum > 0 else np.nan
    else:
        # Weighted: always use actual basket tickers (correct for both active and held days).
        # long_df / short_df are already filtered to basket tickers above.
        long_contrib, short_contrib = 0.0, 0.0
        long_w_sum,   short_w_sum   = 0.0, 0.0
        for sector, grp in long_df.groupby('sector'):
            w_k = sw.get(sector, 0.0)
            if w_k < 1e-6:
                continue
            long_contrib += w_k * grp['target_ret_1d'].mean()
            long_w_sum   += w_k
        for sector, grp in short_df.groupby('sector'):
            w_k = sw.get(sector, 0.0)
            if w_k > -1e-6:
                continue
            short_contrib += w_k * grp['target_ret_1d'].mean()
            short_w_sum   += abs(w_k)
        # short_contrib = Σ(w_k<0)*r_k — negative when shorts rise (bad), positive when they fall (good).
        # This is already in P&L direction, so pass directly as s_pnl.
        l     = long_contrib  / long_w_sum  if long_w_sum  > 0 else 0.0
        s_pnl = short_contrib / short_w_sum if short_w_sum > 0 else 0.0
        port_ret  = _net_exposure_pnl(l, s_pnl, long_alloc, short_alloc)
        long_ret  = l     if long_w_sum  > 0 else np.nan
        short_ret = s_pnl if short_w_sum > 0 else np.nan

    return port_ret, long_ret, short_ret, score_records, new_scores, new_basket

## 8. Walk-Forward Training

In [ ]:
OOS_DIR = SNAPSHOT_DIR / 'oos'
OOS_DIR.mkdir(parents=True, exist_ok=True)

all_ic_results = []
wf_port_rows   = {}
wf_pred_rows   = {}
xgb_imps       = {}
xgb_imps_pos   = {}
xgb_imps_neg   = {}
oos_snap_rows  = {}
sector_ic_history   = {}
sector_sharpe_history = {}

for fold_idx, fold in enumerate(WF_FOLDS):
    fold_name, tr_s, tr_e, te_s, te_e = fold
    print(f'\n── {fold_name} ──')

    train_raw = df[(df['date'] >= tr_s) & (df['date'] <= tr_e)].copy()
    test_raw  = df[(df['date'] >= te_s) & (df['date'] <= te_e)].copy()
    if len(train_raw) < 1000 or len(test_raw) < 100:
        print('  Insufficient data — skipping.'); continue

    X_train, y_train, X_test, train_df, test_df, weights = preprocess_fold(train_raw, test_raw)

    # ── Early stopping validation split (time-based, last ES_VAL_FRAC of train dates) ──
    X_es_val = y_es_val = None
    X_full_es = y_full_es = w_full_es = None
    if EARLY_STOPPING_ROUNDS > 0:
        _udates  = sorted(train_df['date'].unique())
        _n_val   = max(20, int(len(_udates) * ES_VAL_FRAC))
        _val_msk = train_df['date'].isin(set(_udates[-_n_val:])).values
        # Save full arrays before split — used for pass-2 retraining on 100% of data
        X_full_es, y_full_es, w_full_es = X_train, y_train, weights
        X_es_val, y_es_val = X_train[_val_msk], y_train[_val_msk]
        X_train  = X_train[~_val_msk]
        y_train  = y_train[~_val_msk]
        weights  = weights[~_val_msk]
        print(f'  Early stopping: {(~_val_msk).sum():,} train obs / {_val_msk.sum():,} val obs')

    # ── Train global model (always; used as fallback for per-regime and inactive regimes) ──
    if USE_THREE_MODEL:
        global_model = train_xgboost_mixture(X_train, y_train, weights, X_es_val, y_es_val,
                                             X_full_es, y_full_es, w_full_es)
    else:
        global_model = train_xgboost(X_train, y_train, weights, X_es_val, y_es_val,
                                     X_full_es, y_full_es, w_full_es)
    xgb_imps[fold_name] = pd.Series(xgb_feature_importances(global_model), index=FEATURE_COLS)
    if USE_THREE_MODEL:
        xgb_imps_pos[fold_name] = pd.Series(global_model['pos'].feature_importances_, index=FEATURE_COLS)
        xgb_imps_neg[fold_name] = pd.Series(global_model['neg'].feature_importances_, index=FEATURE_COLS)

    # ── OOS loss metrics on test set ───────────────────────────────────────────
    _y_oos   = test_df[TARGET_COL].values.astype(np.float32)
    _y_cls_t = (_y_oos > 0).astype(np.int32)
    if USE_THREE_MODEL:
        _r_pos  = float(np.sqrt(np.mean((_y_oos - global_model['pos'].predict(X_test)) ** 2)))
        _r_neg  = float(np.sqrt(np.mean((_y_oos - global_model['neg'].predict(X_test)) ** 2)))
        _p_pred = global_model['cls'].predict_proba(X_test)[:, 1].clip(1e-7, 1 - 1e-7)
        _ll_oos = float(-np.mean(_y_cls_t * np.log(_p_pred) + (1 - _y_cls_t) * np.log(1 - _p_pred)))
        print(f'    {"pos_reg":18s}  OOS rmse={_r_pos:.5f}')
        print(f'    {"neg_reg":18s}  OOS rmse={_r_neg:.5f}')
        print(f'    {"classifier":18s}  OOS logloss={_ll_oos:.5f}')
    else:
        _r_oos = float(np.sqrt(np.mean((_y_oos - global_model.predict(X_test)) ** 2)))
        print(f'    {"global_reg":18s}  OOS rmse={_r_oos:.5f}')

    _plot_learning_curve(global_model, fold_name)

    # ── Per-regime models (SHARED_XGB=False) ──────────────────────────────────
    if SHARED_XGB:
        regime_models = {r: global_model for r in ACTIVE_REGIMES}
    else:
        regime_models = {}
        for regime in sorted(ACTIVE_REGIMES):
            sub_raw = train_raw[train_raw['regime'] == regime]
            n_dates = sub_raw['date'].nunique()
            if n_dates < MIN_REGIME_DAYS:
                print(f'  XGB [{regime:20s}]  n={n_dates} < {MIN_REGIME_DAYS} → global fallback')
                regime_models[regime] = global_model
            else:
                X_r, y_r, _, w_r = preprocess_train_only(sub_raw)
                if USE_THREE_MODEL:
                    rmodel = train_xgboost_mixture(X_r, y_r, w_r)
                else:
                    rmodel = train_xgboost(X_r, y_r, w_r)
                regime_models[regime] = rmodel
                print(f'  XGB [{regime:20s}]  n_obs={len(y_r):,}')

    # ── Score ALL test rows upfront ────────────────────────────────────────────
    test_df  = test_df.copy()
    preds    = xgb_predict(global_model, X_test)
    if not SHARED_XGB:
        for regime, rmodel in regime_models.items():
            if rmodel is global_model:
                continue
            mask = (test_df['regime'].values == regime)
            if mask.any():
                preds[mask] = xgb_predict(rmodel, X_test[mask])
    test_df['xgb_pred'] = preds

    # IC evaluation (XGBoost scores vs 21d target)
    ic_m = compute_ic_metrics(
        test_df[TARGET_COL].values, test_df['xgb_pred'].values,
        test_df['date'].values,     test_df['ticker'].values)
    all_ic_results.append({'Fold': fold_name,
                           **{k: v for k, v in ic_m.items() if not k.startswith('_')}})
    print(f'  XGBoost  IC={ic_m["Mean Daily IC"]:.4f}  ICIR={ic_m["ICIR"]:.3f}')
    # Tail IC: top/bottom Q% only
    _tic = np.nan; _ticir = np.nan
    _tail_ic_daily = []
    for _td, _tdf in test_df.groupby('date'):
        _tdf = _tdf.dropna(subset=['xgb_pred', TARGET_COL])
        _n = len(_tdf)
        if _n < 10: continue
        _k = max(1, int(_n * QUANTILE_GRID[0]))
        _tail = pd.concat([_tdf.nlargest(_k, 'xgb_pred'), _tdf.nsmallest(_k, 'xgb_pred')])
        if len(_tail) < 4: continue
        _rho, _ = spearmanr(_tail['xgb_pred'], _tail[TARGET_COL])
        _tail_ic_daily.append(_rho)
    if _tail_ic_daily:
        _tic   = float(np.mean(_tail_ic_daily))
        _ticir = _tic / float(np.std(_tail_ic_daily)) if np.std(_tail_ic_daily) > 1e-8 else np.nan
        print(f'  Tail IC (Q={QUANTILE_GRID[0]:.0%})  IC={_tic:.4f}  ICIR={_ticir:.3f}')

    QUANTILE = QUANTILE_GRID[0]   # diagnostic table uses first grid value only

    # ── Pass-1 vs Pass-2 OOS comparison table (when two-pass ES is active) ────
    if EARLY_STOPPING_ROUNDS > 0 and X_full_es is not None:
        # Retrieve pass-1 model and get its test predictions
        if USE_THREE_MODEL:
            _p1 = {k: getattr(v, '_pass1_model', None) for k, v in global_model.items()}
            _has_p1 = all(m is not None for m in _p1.values())
            p1_pred = predict_xgboost_mixture(_p1, X_test).astype(np.float64) if _has_p1 else None
        else:
            _p1_m = getattr(global_model, '_pass1_model', None)
            p1_pred = _p1_m.predict(X_test).astype(np.float64) if _p1_m is not None else None

        if p1_pred is not None:
            p2_pred    = test_df['xgb_pred'].values
            y_tgt      = test_df[TARGET_COL].values
            r1d        = test_df['target_ret_1d'].values
            dates_test = test_df['date'].values

            def _fold_ic(pred):
                ic_g, _ = spearmanr(pred, y_tgt)
                tmp = pd.DataFrame({'d': dates_test, 'p': pred, 'y': y_tgt})
                di  = tmp.groupby('d').apply(
                    lambda g: spearmanr(g['p'], g['y'])[0] if len(g) >= 5 else np.nan).dropna()
                return float(ic_g), float(di.mean()), float(di.std()), float(di.mean() / di.std()) if di.std() > 0 else np.nan

            def _fold_ls(pred):
                tmp = pd.DataFrame({'d': dates_test, 'p': pred, 'r': r1d})
                dr  = []
                for _, g in tmp.groupby('d'):
                    g = g.dropna()
                    n = len(g)
                    if n < 10: continue
                    np_ = max(1, int(np.ceil(n * QUANTILE)))
                    s   = g.sort_values('p')
                    dr.append((s.iloc[-np_:]['r'].mean() - s.iloc[:np_]['r'].mean()) / 2)
                if len(dr) < 10: return np.nan, np.nan, np.nan
                r = np.array(dr)
                return float(r.mean() * 252), float(r.std() * np.sqrt(252)), float(r.mean() / r.std() * np.sqrt(252)) if r.std() > 0 else np.nan

            ic1_g, ic1_m, ic1_s, icir1 = _fold_ic(p1_pred)
            ic2_g, ic2_m, ic2_s, icir2 = _fold_ic(p2_pred)
            ann1, vol1, sh1 = _fold_ls(p1_pred)
            ann2, vol2, sh2 = _fold_ls(p2_pred)

            _W = 18
            print(f'\n  {"":30s}  {"Pass1 (85%)":>{_W}s}  {"Pass2 (100%)":>{_W}s}')
            print(f'  {"-"*68}')
            for nm, v1, v2 in [
                ('IC (global)',      ic1_g,  ic2_g),
                ('Mean Daily IC',    ic1_m,  ic2_m),
                ('Daily IC Std',     ic1_s,  ic2_s),
                ('ICIR',             icir1,  icir2),
                ('L/S Ann. Return',  ann1,   ann2),
                ('L/S Ann. Vol',     vol1,   vol2),
                ('L/S Sharpe',       sh1,    sh2),
            ]:
                s1 = f'{v1:{_W}.4f}' if np.isfinite(v1) else f'{"n/a":>{_W}s}'
                s2 = f'{v2:{_W}.4f}' if np.isfinite(v2) else f'{"n/a":>{_W}s}'
                print(f'  {nm:30s}  {s1}  {s2}')
            print()

    # ── Sector weights per active regime ──────────────────────────────────────
    # Always compute in-sample XGBoost scores so eval_regime_train_sharpe can use them.
    # Use X_full_es (100% of training rows) when ES split was active; X_train is 85% after split.
    train_df   = train_df.copy()
    X_tr_score = X_full_es if (EARLY_STOPPING_ROUNDS > 0 and X_full_es is not None) else X_train
    scores     = xgb_predict(global_model, X_tr_score)
    if not SHARED_XGB:
        for regime, rmodel in regime_models.items():
            if rmodel is global_model:
                continue
            mask = (train_df['regime'].values == regime)
            if mask.any():
                scores[mask] = xgb_predict(rmodel, X_tr_score[mask])
    train_df['xgb_score'] = scores

    for _q in QUANTILE_GRID:
        # Store THIS fold's IC with end date; use only prior folds for weights (no look-ahead)
        sector_ic_history[(fold_idx, _q)] = {
            'ic':       compute_fold_sector_ic(test_df, _q),
            'end_date': te_e,
        }
        _weighted_ic = get_weighted_sector_ic(sector_ic_history, fold_idx, _q, te_s)
        _raw_sec_ic = sector_ic_history[(fold_idx, _q)]['ic']
        _raw_ic_str = ''.join(f'{s}:{v:+.3f}  ' for s, v in sorted(_raw_sec_ic.items(), key=lambda x: -x[1])).rstrip()
        print(f'  Sector IC (Q={_q:.0%}): {_raw_ic_str}')
        _raw_reg_ic = {}
        for _reg in sorted(ACTIVE_REGIMES):
            _rsub = test_df[test_df['regime'] == _reg]
            _rdaily = []
            for _, _rdg in _rsub.groupby('date'):
                _rn = len(_rdg.dropna(subset=['xgb_pred', TARGET_COL]))
                if _rn < 5: continue
                _rk = max(1, int(_rn * _q))
                _rtail = pd.concat([_rdg.nlargest(_rk, 'xgb_pred'), _rdg.nsmallest(_rk, 'xgb_pred')])
                if len(_rtail) < 4: continue
                _rho, _ = spearmanr(_rtail['xgb_pred'], _rtail[TARGET_COL])
                _rdaily.append(_rho)
            if _rdaily:
                _raw_reg_ic[_reg] = float(np.mean(_rdaily))
        _reg_ic_str = '  '.join(f'{r}:{v:+.3f}' for r, v in sorted(_raw_reg_ic.items(), key=lambda x: -x[1]))
        print(f'  Regime IC  (Q={_q:.0%}): {_reg_ic_str}')
        _raw_sec_sharpes = compute_fold_sector_sharpes(test_df, _q)
        sector_sharpe_history[(fold_idx, _q)] = {'sharpes': _raw_sec_sharpes, 'end_date': te_e}
        _wl_sharpes, _ws_sharpes = get_weighted_sector_sharpes(
            sector_sharpe_history, fold_idx, _q, te_s)
        for _lam in FIXED_LAM_GRID:
            for _use_mw in MARKOWITZ_GRID:
                for _gross, _net in EXPOSURE_GRID:
                    GROSS_EXPOSURE = _gross
                    NET_EXPOSURE   = _net
                    QUANTILE      = _q
                    FIXED_LAM     = _lam
                    USE_MARKOWITZ = _use_mw
                    print(f'\n  ── Q={_q:.0%}  LAM={_lam}  MW={_use_mw}  G={_gross}  N={_net} ──')
                    _ic_lines = '  '.join(f'{s}:{v:+.3f}' for s, v in sorted(_weighted_ic.items(), key=lambda x: -x[1]))
                    print(f'    IC weights (fold {fold_idx}, Q={_q:.0%}): {_ic_lines if _ic_lines else "no prior IC — equal weight"}')

                    regime_sw = {}
                    for regime in sorted(ACTIVE_REGIMES):
                        sub     = train_df[train_df['regime'] == regime]
                        n_dates = sub['date'].nunique()

                        if SECTOR_WEIGHTING == 'none':
                            sw = None
                        elif n_dates < MIN_REGIME_DAYS:
                            print(f'  {regime:25s}  n={n_dates} < {MIN_REGIME_DAYS} → equal weight fallback')
                            sw = None
                        else:
                            sw = get_sector_weights(sub, weighted_ic=_weighted_ic,
                                            weighted_long_sharpes=_wl_sharpes,
                                            weighted_short_sharpes=_ws_sharpes)

                        regime_sw[regime] = sw

                        tr_sharpe = eval_regime_train_sharpe(sub, sw)
                        sh_str    = f'{tr_sharpe:.3f}' if not np.isnan(tr_sharpe) else 'n/a'
                        if isinstance(sw, tuple):
                            _lsw, _ssw = sw
                            if _lsw is None and _ssw is None:
                                print(f'  {regime:25s}  n={n_dates}  sharpe={sh_str}  (IC fallback: equal weight)')
                            else:
                                _l_str = '  '.join(f'{s}({w:.0%})' for s, w in sorted((_lsw or {}).items(), key=lambda x: -x[1]))
                                _s_str = '  '.join(f'{s}({w:.0%})' for s, w in sorted((_ssw or {}).items(), key=lambda x: -x[1]))
                                print(f'  {regime:25s}  n={n_dates}  sharpe={sh_str}')
                                if _l_str: print(f'    L: {_l_str}')
                                if _s_str: print(f'    S: {_s_str}')
                        elif sw is not None:
                            n_long  = sum(1 for v in sw.values() if v > 1e-6)
                            n_short = sum(1 for v in sw.values() if v < -1e-6)
                            long_secs  = [s for s, v in sw.items() if v >  1e-6]
                            short_secs = [s for s, v in sw.items() if v < -1e-6]
                            _l_str = '  '.join(f'{s}({v:.0%})' for s, v in sorted(sw.items(), key=lambda x: -x[1]) if v > 1e-6)
                            _s_str = '  '.join(f'{s}({abs(v):.0%})' for s, v in sorted(sw.items(), key=lambda x: x[1]) if v < -1e-6)
                            print(f'  {regime:25s}  n={n_dates}  sharpe={sh_str}  long={n_long}  short={n_short}')
                            if _l_str: print(f'    L: {_l_str}')
                            if _s_str: print(f'    S: {_s_str}')
                        else:
                            print(f'  {regime:25s}  n={n_dates}  sharpe={sh_str}  (equal weight)')

                    # ── Day-by-day portfolio construction ─────────────────────────────────────
                    # Compute fold-level L/S allocations (Markowitz or fixed global)
                    if USE_MARKOWITZ:
                        fold_la, fold_sa, _mu_l, _mu_s = compute_markowitz_alloc(train_df, regime_sw)
                        print(f'  Markowitz alloc: long={fold_la:.3f}  short={fold_sa:.3f}  net={fold_la-fold_sa:.3f}  (mu_l={_mu_l:.5f}  mu_s={_mu_s:.5f})')
                    else:
                        fold_la, fold_sa = _default_allocs()
                        print(f'  Default alloc:   long={fold_la:.3f}  short={fold_sa:.3f}')

                    held_scores         = {}
                    last_basket         = None
                    held_sector_weights = None
                    fold_port_rows      = []
                    fold_pred_rows      = []

                    for date, date_df in test_df.groupby('date'):
                        regime      = date_df['regime'].iloc[0]
                        regime_base = date_df['regime_base'].iloc[0]
                        is_active   = regime in ACTIVE_REGIMES

                        if not is_active and last_basket is None:
                            continue   # no position yet

                        is_held = not is_active
                        sw = regime_sw.get(regime, None) if is_active else held_sector_weights

                        _brake   = brake_scale_map.get(pd.Timestamp(date), 1.0) if USE_EXPOSURE_BRAKE else 1.0
                        _date_la = fold_la * _brake
                        port_ret, long_ret, short_ret, score_recs, held_scores, cur_basket =                         build_date_portfolio(date_df, held_scores, last_basket, is_active, sw,
                                                 _date_la, fold_sa)

                        if is_active:
                            last_basket         = cur_basket
                            held_sector_weights = sw

                        # OOS snapshot: sector→ticker mapping from score_recs
                        ticker_sector = date_df.set_index('ticker')['sector'].to_dict()
                        sec_tickers   = {}
                        for rec in score_recs:
                            s = ticker_sector.get(rec['ticker'], 'Unknown')
                            sec_tickers.setdefault(s, []).append(rec['ticker'])
                        oos_snap_rows.setdefault((_q, _lam, _use_mw, _gross, _net), []).append({
                            'date':             date,
                            'fold':             fold_name,
                            'regime':           regime,
                            'is_held':          is_held,
                            'port_ret':         port_ret,
                            'sector_weighting': SECTOR_WEIGHTING,
                            **{f'w_{s}': (((sw[0] or {}).get(s, 0.0) - (sw[1] or {}).get(s, 0.0)) if isinstance(sw, tuple) else (sw.get(s, 0.0) if sw else 0.0)) for s in ALL_SECTORS},
                            **{s: sec_tickers.get(s, [])                  for s in ALL_SECTORS},
                        })

                        row = {
                            'date': date, 'fold': fold_name,
                            'regime': regime, 'regime_base': regime_base,
                            'is_held': is_held,
                            'port_ret': port_ret, 'long_ret': long_ret, 'short_ret': short_ret,
                            'long_alloc': _date_la, 'short_alloc': fold_sa, 'brake_scale': _brake,
                        }
                        _bkt = cur_basket if cur_basket else {'long': set(), 'short': set()}
                        _lt  = _bkt.get('long',  set())
                        _st  = _bkt.get('short', set())
                        _total_long  = len(_lt)
                        _total_short = len(_st)
                        for _sec in ALL_SECTORS:
                            _sg  = date_df[date_df['sector'] == _sec][['ticker', 'target_ret_1d']].dropna(subset=['target_ret_1d'])
                            _lg  = _sg[_sg['ticker'].isin(_lt)]['target_ret_1d']
                            _sg2 = _sg[_sg['ticker'].isin(_st)]['target_ret_1d']
                            row[f'l_{_sec}'] = float(_lg.mean())  if len(_lg)  > 0 else np.nan
                            row[f's_{_sec}'] = float(_sg2.mean()) if len(_sg2) > 0 else np.nan
                            row[f'lw_{_sec}'] = float(_lg.mean()  * len(_lg)  / _total_long)  if len(_lg)  > 0 and _total_long  > 0 else np.nan
                            row[f'sw_{_sec}'] = float(_sg2.mean() * len(_sg2) / _total_short) if len(_sg2) > 0 and _total_short > 0 else np.nan
                            _sg_e = _sg.copy()
                            _sg_e['_ema'] = _sg_e['ticker'].map(held_scores)
                            _sg_e = _sg_e.dropna(subset=['_ema'])
                            if len(_sg_e) >= 5:
                                _k = max(1, int(np.ceil(len(_sg_e) * QUANTILE)))
                                row[f'ul_{_sec}'] = float(_sg_e.nlargest(_k,  '_ema')['target_ret_1d'].mean())
                                row[f'us_{_sec}'] = float(_sg_e.nsmallest(_k, '_ema')['target_ret_1d'].mean())
                            else:
                                row[f'ul_{_sec}'] = np.nan
                                row[f'us_{_sec}'] = np.nan
                        wf_port_rows.setdefault((_q, _lam, _use_mw, _gross, _net), []).append(row)
                        fold_port_rows.append(row)
                        if not is_held:
                            for rec in score_recs:
                                r = {'date': date, 'fold': fold_name, **rec}
                                wf_pred_rows.setdefault((_q, _lam, _use_mw, _gross, _net), []).append(r)
                                fold_pred_rows.append(r)

                    # ── End-of-fold OOS summary ───────────────────────────────────────────
                    all_dates = test_df['date'].nunique()
                    if fold_port_rows:
                        _fp_df   = pd.DataFrame(fold_port_rows)
                        n_active = int((~_fp_df['is_held']).sum())
                        print(f'  Allocation  long={fold_la:.3f}×  short={fold_sa:.3f}×  gross={fold_la+fold_sa:.3f}×  net={fold_la-fold_sa:.3f}×  active={n_active}/{all_dates}')

                        _MCOLS = ['AnnRet', 'AnnVol', 'Sharpe', 'Sortino', 'Calmar', 'MaxDD', 'WinRate', 'VaR5%', 'CVaR5%']
                        _CW    = [26] + [9] * len(_MCOLS)
                        _HDR   = '  ' + '  '.join(f'{c:>{_CW[i]}}' for i, c in enumerate([''] + _MCOLS))

                        def _mf(v, pct=False):
                            if not np.isfinite(float(v)): return 'n/a'
                            return f'{v:+.1%}' if pct else f'{v:+.3f}'

                        def _mrow(label, rets):
                            _s = pd.Series(rets).dropna()
                            if len(_s) < 5: return f'  {label:<26s}  (insufficient data)'
                            m = compute_strategy_metrics(_s)
                            vals = [label, _mf(m['Ann. Return'],True), _mf(m['Ann. Vol'],True),
                                    _mf(m['Sharpe']), _mf(m['Sortino']), _mf(m['Calmar']),
                                    _mf(m['Max DD'],True), _mf(m['Win Rate'],True),
                                    _mf(m['VaR 5%'],True), _mf(m['CVaR 5%'],True)]
                            return '  ' + '  '.join(f'{v:>{_CW[i]}}' for i, v in enumerate(vals))

                        # ── Portfolio breakdown ──────────────────────────────────────────
                        print(f'\n  ── Portfolio Metrics ─────────────────────────────────────────────────')
                        print(_HDR)
                        print(_mrow('Overall',    _fp_df['port_ret'].values))
                        print(_mrow('Active',     _fp_df[~_fp_df['is_held']]['port_ret'].values))
                        print(_mrow('Held',       _fp_df[_fp_df['is_held']]['port_ret'].values))
                        print(_mrow('Long Book',  _fp_df['long_ret'].values))
                        print(_mrow('Short Book', _fp_df['short_ret'].values))

                        # ── IC metrics ──────────────────────────────────────────────────
                        if fold_pred_rows:
                            _fp   = pd.DataFrame(fold_pred_rows)
                            _icm  = compute_ic_metrics(_fp['y_true'].values, _fp['y_pred'].values,
                                                       _fp['date'].values, _fp['ticker'].values)
                            _tic_v   = _tic   if '_tic'   in vars() and np.isfinite(_tic)   else np.nan
                            _ticir_v = _ticir if '_ticir' in vars() and np.isfinite(_ticir) else np.nan
                            print(f'\n  ── IC Metrics ────────────────────────────────────────────────────────')
                            print(f'  {"Global IC":<25s}  IC={_icm["Mean Daily IC"]:+.4f}   ICIR={_icm["ICIR"]:+.4f}')
                            print(f'  {f"Tail IC (Q={_q:.0%})":<25s}  IC={_tic_v:+.4f}   ICIR={_ticir_v:+.4f}')

                        # ── Regime table ────────────────────────────────────────────────
                        print(f'\n  ── Regime Metrics ────────────────────────────────────────────────────')
                        print(_HDR)
                        for _reg in sorted(_fp_df['regime'].unique()):
                            _rdf = _fp_df[_fp_df['regime'] == _reg]
                            if len(_rdf) >= 5:
                                print(_mrow(_reg, _rdf['port_ret'].values))

                        # ── Sector tables ───────────────────────────────────────────────
                        print(f'\n  ── Sector Attribution (Net Portfolio) ───────────────────────────────')
                        print(_HDR)
                        for _sec in ALL_SECTORS:
                            _lwc = f'lw_{_sec}'
                            _swc = f'sw_{_sec}'
                            if _lwc not in _fp_df.columns and _swc not in _fp_df.columns: continue
                            _la  = _fp_df['long_alloc'].iloc[0]
                            _sa  = _fp_df['short_alloc'].iloc[0]
                            _lwr = _fp_df[_lwc].fillna(0)
                            _swr = _fp_df[_swc].fillna(0)
                            _net = _la * _lwr - _sa * _swr
                            _net = _net[(_fp_df[_lwc].notna()) | (_fp_df[_swc].notna())]
                            if len(_net) >= 5:
                                print(_mrow(_sec, _net.values))

                        print(f'\n  ── Sector Metrics (Long Leg) ─────────────────────────────────────────')
                        print(_HDR)
                        for _sec in ALL_SECTORS:
                            _lc = f'l_{_sec}'
                            if _lc not in _fp_df.columns: continue
                            _lr = _fp_df[_lc].dropna()
                            if len(_lr) >= 5:
                                print(_mrow(_sec, _lr.values))
                            _lwc = f'lw_{_sec}'
                            if _lwc in _fp_df.columns:
                                _lwr = _fp_df[_lwc].dropna()
                                if len(_lwr) >= 5:
                                    print(_mrow(f'{_sec} ($-wtd)', _lwr.values))

                        print(f'\n  ── Sector Metrics (Short Leg) ────────────────────────────────────────')
                        print(_HDR)
                        for _sec in ALL_SECTORS:
                            _sc = f's_{_sec}'
                            if _sc not in _fp_df.columns: continue
                            _sr = _fp_df[_sc].dropna()
                            if len(_sr) >= 5:
                                print(_mrow(_sec, (-_sr).values))
                            _swc = f'sw_{_sec}'
                            if _swc in _fp_df.columns:
                                _swr = _fp_df[_swc].dropna()
                                if len(_swr) >= 5:
                                    print(_mrow(f'{_sec} ($-wtd)', (-_swr).values))

print(f'\nCombos evaluated: {len(wf_port_rows)}')
print(f'Total combo×day rows: {sum(len(v) for v in wf_port_rows.values())}')

for (_q, _lam, _use_mw, _gross, _net), snap_rows in oos_snap_rows.items():
    if snap_rows:
        snap_df = pd.DataFrame(snap_rows)
        snap_df['date'] = pd.to_datetime(snap_df['date'])
        fname = f'oos_holdings_q{int(_q*100)}_lam{int(_lam*100)}_mw{int(_use_mw)}_g{int(_gross*10)}_n{int(_net*10)}.parquet'
        snap_df.to_parquet(OOS_DIR / fname, index=False)
print(f'Saved {len(oos_snap_rows)} OOS snapshots → {OOS_DIR}')

## 9. Analysis

In [ ]:
# ── 9a. IC Summary ────────────────────────────────────────────────────────────
ic_df = pd.DataFrame(all_ic_results)
print('=== Walk-Forward IC Summary (XGBoost vs target_ret_21d) ===')
print(ic_df[['Fold', 'Mean Daily IC', 'ICIR', 'Hit Rate', 'R2']].to_string(index=False))
print(f'\nMean IC={ic_df["Mean Daily IC"].mean():.4f}  Mean ICIR={ic_df["ICIR"].mean():.4f}')

In [ ]:
# ── 9b. XGBoost Feature Importance ───────────────────────────────────────────
if USE_THREE_MODEL:
    imp_cls = pd.DataFrame(xgb_imps).mean(axis=1).sort_values(ascending=False)
    imp_pos = pd.DataFrame(xgb_imps_pos).mean(axis=1).sort_values(ascending=False)
    imp_neg = pd.DataFrame(xgb_imps_neg).mean(axis=1).sort_values(ascending=False)
    fig, axes = plt.subplots(1, 3, figsize=(18, 8))
    for ax, imp, title, col in zip(axes,
            [imp_cls, imp_pos, imp_neg],
            ['Classifier (P(y>0))', 'Pos Regressor (y>0)', 'Neg Regressor (y<=0)'],
            ['steelblue', 'seagreen', 'tomato']):
        ax.barh(imp.index[:20][::-1], imp.values[:20][::-1], color=col, alpha=0.85)
        ax.set_title(title, fontsize=9)
        ax.set_xlabel('Importance')
    plt.suptitle('XGBoost Three-Model Feature Importance (top 20, WF avg)', fontsize=10)
    plt.tight_layout(); plt.show()
    print('\nTop 15 — Classifier:')
    print(imp_cls.head(15).round(5).to_string())
    print('\nTop 15 — Pos Regressor:')
    print(imp_pos.head(15).round(5).to_string())
    print('\nTop 15 — Neg Regressor:')
    print(imp_neg.head(15).round(5).to_string())
else:
    imp_mean = pd.DataFrame(xgb_imps).mean(axis=1).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(imp_mean.index[:25][::-1], imp_mean.values[:25][::-1],
            color='steelblue', alpha=0.85)
    ax.set_title('XGBoost Mean Feature Importance (top 25, WF avg)')
    ax.set_xlabel('Importance')
    plt.tight_layout(); plt.show()
    print('\nTop 15:')
    print(imp_mean.head(15).round(5).to_string())

# ── Export feature importances ────────────────────────────────────────────────
_imp_out = BASE_DIR / 'feature_importances.csv'
if USE_THREE_MODEL:
    _imp_df = pd.DataFrame({
        'classifier':    pd.DataFrame(xgb_imps).mean(axis=1),
        'pos_regressor': pd.DataFrame(xgb_imps_pos).mean(axis=1),
        'neg_regressor': pd.DataFrame(xgb_imps_neg).mean(axis=1),
    })
    _imp_df['mean'] = _imp_df.mean(axis=1)
    _imp_df = _imp_df.sort_values('mean', ascending=False)
else:
    _imp_df = pd.DataFrame(xgb_imps).mean(axis=1).rename('importance').sort_values(ascending=False).to_frame()
_imp_df.index.name = 'feature'
_imp_df.to_csv(_imp_out)
print(f'\nFeature importances exported → {_imp_out}')

In [ ]:
# ── 9c. Per-combo evaluation ─────────────────────────────────────────────────
def add_regime_background(ax, dates, regime_series):
    dt = pd.DatetimeIndex(dates)
    reg = regime_series.reindex(dt).ffill()
    prev, start = None, dt[0]
    for i, r in enumerate(reg):
        if r != prev:
            if prev is not None and str(prev) in REGIME_COLORS:
                ax.axvspan(start, dt[i], alpha=0.18, color=REGIME_COLORS[str(prev)], lw=0, zorder=0)
            prev, start = r, dt[i]
    if prev is not None and str(prev) in REGIME_COLORS:
        ax.axvspan(start, dt[-1], alpha=0.18, color=REGIME_COLORS[str(prev)], lw=0, zorder=0)

def shade_held(ax, dates, is_held_series):
    dt = pd.DatetimeIndex(dates)
    in_held, start = False, dt[0]
    for i, h in enumerate(is_held_series.reindex(dt).fillna(False)):
        if h and not in_held:   start, in_held = dt[i], True
        elif not h and in_held: ax.axvspan(start, dt[i], alpha=0.12, color='gray', lw=0, zorder=1); in_held = False
    if in_held: ax.axvspan(start, dt[-1], alpha=0.12, color='gray', lw=0, zorder=1)

# ── Metric table helpers ─────────────────────────────────────────────────────
_LW = 28
_CW = 10
_HDR_COLS = ['AnnRet', 'AnnVol', 'Sharpe', 'Sortino', 'Calmar', 'MaxDD', 'WinRate', 'VaR5%', 'CVaR5%']

def _fp(v, dec=1):
    if v is None or (isinstance(v, float) and np.isnan(v)): return f"{'n/a':>{_CW}}"
    return f"{f'{v*100:+.{dec}f}%':>{_CW}}"

def _ff(v, dec=3):
    if v is None or (isinstance(v, float) and np.isnan(v)): return f"{'n/a':>{_CW}}"
    return f"{f'{v:+.{dec}f}':>{_CW}}"

def _hdr():
    return ' ' * _LW + ''.join(f"{c:>{_CW}}" for c in _HDR_COLS)

def _row(label, m):
    ann = m.get('Ann. Return', np.nan) if m else np.nan
    if isinstance(ann, float) and np.isnan(ann):
        return f"  {label:<{_LW - 2}}(insufficient data)"
    return (f"{label:>{_LW}}"
            + _fp(m.get('Ann. Return', np.nan))
            + _fp(m.get('Ann. Vol',    np.nan))
            + _ff(m.get('Sharpe',      np.nan))
            + _ff(m.get('Sortino',     np.nan))
            + _ff(m.get('Calmar',      np.nan))
            + _fp(m.get('Max DD',      np.nan))
            + _fp(m.get('Win Rate',    np.nan))
            + _fp(m.get('VaR 5%',     np.nan))
            + _fp(m.get('CVaR 5%',    np.nan)))

def _compute_tail_ic(pred_df, q):
    from scipy.stats import spearmanr as _sp
    daily = []
    for _, g in pred_df.groupby('date'):
        if len(g) < 10: continue
        k = max(1, int(len(g) * q))
        tail = pd.concat([g.nlargest(k, 'y_pred'), g.nsmallest(k, 'y_pred')])
        if len(tail) < 2: continue
        ic, _ = _sp(tail['y_pred'], tail['y_true'])
        daily.append(ic)
    s = pd.Series(daily).dropna()
    if len(s) < 2: return np.nan, np.nan
    return float(s.mean()), float(s.mean() / s.std()) if s.std() > 1e-8 else np.nan

_REGIME_FINE = ['Extreme Fear', 'Fear-Falling', 'Fear-Rising', 'Neutral',
                'Greed-Rising', 'Greed-Falling', 'Extreme Greed']


for (_q, _lam, _use_mw, _gross, _net), port_rows in wf_port_rows.items():
    print(f'\n{"="*60}')
    print(f'Q={_q:.0%}  LAM={_lam}  MW={_use_mw}  G={_gross}  N={_net}')
    print(f'{"="*60}')
    wf_port_df = (pd.DataFrame(port_rows)
                  .assign(date=lambda x: pd.to_datetime(x['date']))
                  .set_index('date').sort_index())
    wf_pred_df = pd.DataFrame(wf_pred_rows[(_q, _lam, _use_mw, _gross, _net)])

    _la_s  = wf_port_df['long_alloc'].values
    _sa_s  = wf_port_df['short_alloc'].values
    _s_raw = (-wf_port_df['short_ret']) if SECTOR_WEIGHTING == 'none' else wf_port_df['short_ret']
    _l_ser = (wf_port_df['long_ret']  * wf_port_df['long_alloc']).dropna()
    _s_ser = (_s_raw * wf_port_df['short_alloc']).dropna()

    # ── Portfolio Metrics ──────────────────────────────────────────────────
    print(f'\n  {"─ Portfolio Metrics ":-<60}')
    print(_hdr())
    print(_row('Overall',   compute_strategy_metrics(wf_port_df['port_ret'])))
    print(_row('Active',    compute_strategy_metrics(wf_port_df.loc[~wf_port_df['is_held'], 'port_ret'])))
    print(_row('Held',      compute_strategy_metrics(wf_port_df.loc[ wf_port_df['is_held'], 'port_ret'])))
    print(_row('Long Book',  compute_strategy_metrics(_l_ser)))
    print(_row('Short Book', compute_strategy_metrics(_s_ser)))

    # ── IC Metrics ────────────────────────────────────────────────────────
    print(f'\n  ── IC Metrics {"─"*46}')
    _ic_g, _icir_g = ic_df['Mean Daily IC'].mean(), ic_df['ICIR'].mean()
    print(f"  {'Global IC':<27}  IC={_ic_g:+.4f}   ICIR={_icir_g:+.4f}")
    if len(wf_pred_df) > 0 and 'y_pred' in wf_pred_df.columns:
        _tic, _ticir = _compute_tail_ic(wf_pred_df, _q)
        print(f"  {f'Tail IC (Q={_q:.0%})':<27}  IC={_tic:+.4f}   ICIR={_ticir:+.4f}")

    # ── Regime Metrics ────────────────────────────────────────────────────
    print(f'\n  ── Regime Metrics {"─"*42}')
    print(_hdr())
    for _reg in _REGIME_FINE:
        if _reg not in wf_port_df['regime'].values: continue
        _sub = wf_port_df[wf_port_df['regime'] == _reg]['port_ret']
        if len(_sub) < 5: continue
        print(_row(_reg, compute_strategy_metrics(_sub)))

    # ── Sector Attribution (Net Portfolio) ───────────────────────────────
    _all_sec = [s for s in ALL_SECTORS
                if f'l_{s}' in wf_port_df.columns or f's_{s}' in wf_port_df.columns]
    if _all_sec:
        print(f'\n  ── Sector Attribution (Net Portfolio) {"─"*22}')
        print(_hdr())
        for _sec in _all_sec:
            _lv = wf_port_df[f'l_{_sec}'].values if f'l_{_sec}' in wf_port_df.columns else np.full(len(wf_port_df), np.nan)
            _sv = wf_port_df[f's_{_sec}'].values if f's_{_sec}' in wf_port_df.columns else np.full(len(wf_port_df), np.nan)
            _nv = np.where(~np.isnan(_lv), _lv * _la_s,
                  np.where(~np.isnan(_sv), -_sv * _sa_s, np.nan))
            _ns = pd.Series(_nv, index=wf_port_df.index).dropna()
            if len(_ns) >= 5:
                print(_row(_sec, compute_strategy_metrics(_ns)))

        # ── Sector Metrics (Long Leg) ─────────────────────────────────
        _long_secs = [s for s in ALL_SECTORS
                      if f'ul_{s}' in wf_port_df.columns
                      and f'l_{s}' in wf_port_df.columns
                      and wf_port_df[f'l_{s}'].notna().any()]
        if _long_secs:
            print(f'\n  ── Sector Metrics (Long Leg) {"─"*31}')
            print(_hdr())
            for _sec in _long_secs:
                _lv = pd.Series(wf_port_df[f'ul_{_sec}'].values, index=wf_port_df.index).dropna()
                if len(_lv) >= 5:
                    print(_row(_sec, compute_strategy_metrics(_lv)))
                    _wtd = (_lv * pd.Series(_la_s, index=wf_port_df.index).reindex(_lv.index)).dropna()
                    if len(_wtd) >= 5:
                        print(_row(f'{_sec} ($-wtd)', compute_strategy_metrics(_wtd)))

        # ── Sector Metrics (Short Leg) ────────────────────────────────
        _short_secs = [s for s in ALL_SECTORS
                       if f'us_{s}' in wf_port_df.columns
                       and f's_{s}' in wf_port_df.columns
                       and wf_port_df[f's_{s}'].notna().any()]
        if _short_secs:
            print(f'\n  ── Sector Metrics (Short Leg) {"─"*30}')
            print(_hdr())
            for _sec in _short_secs:
                _sv = pd.Series(-wf_port_df[f'us_{_sec}'].values, index=wf_port_df.index).dropna()
                if len(_sv) >= 5:
                    print(_row(_sec, compute_strategy_metrics(_sv)))
                    _wtd = (_sv * pd.Series(_sa_s, index=wf_port_df.index).reindex(_sv.index)).dropna()
                    if len(_wtd) >= 5:
                        print(_row(f'{_sec} ($-wtd)', compute_strategy_metrics(_wtd)))

    # ── Equity Curves ────────────────────────────────────────────────────────
    dates   = wf_port_df.index
    s_ls    = wf_port_df['port_ret'].fillna(0)
    s_long  = wf_port_df['long_ret'].fillna(0)
    s_short = wf_port_df['short_ret'].fillna(0)

    fig, (eq_ax, dd_ax) = plt.subplots(2, 1, figsize=(14, 6),
                                        gridspec_kw={'height_ratios': [3, 1]})
    add_regime_background(eq_ax, dates, fng_regime_series)
    shade_held(eq_ax, dates, wf_port_df['is_held'])
    eq_ax.plot(dates, (1+s_ls).cumprod().values,    color='black',     lw=1.5, zorder=5, label='L/S')
    eq_ax.plot(dates, (1+s_long).cumprod().values,  color='steelblue', lw=1.0, zorder=4, alpha=0.8, label='Long')
    eq_ax.plot(dates, (1+s_short).cumprod().values, color='tomato',    lw=1.0, zorder=4, alpha=0.8, label='Short')
    eq_ax.axhline(1, color='gray', lw=0.7, ls='--')
    for _, _, _, te_s, _ in WF_FOLDS:
        eq_ax.axvline(te_s, color='black', lw=0.8, ls=':', alpha=0.5, zorder=6)
    import matplotlib.dates as mdates
    eq_ax.xaxis.set_major_locator(mdates.YearLocator())
    eq_ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    eq_ax.set_title(f'Hybrid XGBoost [{SECTOR_WEIGHTING}] Q={_q:.0%} LAM={_lam} MW={_use_mw} — grey=held', fontsize=10)
    eq_ax.set_ylabel('Cumul. Return')
    patches = [mpatches.Patch(color=c, alpha=0.5, label=r) for r, c in REGIME_COLORS.items()]
    eq_ax.legend(handles=patches + [
        plt.Line2D([0],[0], color='black',     lw=1.5, label='L/S'),
        plt.Line2D([0],[0], color='steelblue', lw=1.0, label='Long'),
        plt.Line2D([0],[0], color='tomato',    lw=1.0, label='Short'),
        plt.Line2D([0],[0], color='black',     lw=0.8, ls=':', alpha=0.5, label='Retrain'),
    ], fontsize=7, loc='upper left', ncol=2)
    cum_ls = (1+s_ls).cumprod()
    dd = (cum_ls - cum_ls.cummax()) / cum_ls.cummax()
    dd_ax.fill_between(dates, dd.values, 0, color='tomato', alpha=0.5)
    dd_ax.xaxis.set_major_locator(mdates.YearLocator())
    dd_ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    for _, _, _, te_s, _ in WF_FOLDS:
        dd_ax.axvline(te_s, color='black', lw=0.8, ls=':', alpha=0.5)
    dd_ax.set_ylabel('Drawdown'); dd_ax.set_xlabel('')
    plt.tight_layout(); plt.show()

    # ── Monthly Heatmap ──────────────────────────────────────────────────────
    ls_monthly = wf_port_df['port_ret'].resample('ME').sum()
    ls_monthly.index = ls_monthly.index.to_period('M')
    pivot = ls_monthly.to_frame('ret')
    pivot['year']  = pivot.index.year
    pivot['month'] = pivot.index.month
    heat = pivot.pivot(index='year', columns='month', values='ret')
    heat.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec'][:len(heat.columns)]
    fig, ax = plt.subplots(figsize=(13, 5))
    import seaborn as sns
    sns.heatmap(heat, ax=ax, cmap='RdYlGn', center=0,
                annot=True, fmt='.3f', linewidths=0.4, cbar_kws={'shrink': 0.6})
    ax.set_title(f'Monthly L/S Returns — Hybrid [{SECTOR_WEIGHTING}] Q={_q:.0%} LAM={_lam} MW={_use_mw}', fontsize=10)
    plt.tight_layout(); plt.show()

    # ── Rolling Sharpe ───────────────────────────────────────────────────────
    ls_s = wf_port_df['port_ret'].fillna(0)
    roll = (ls_s.rolling(63).mean() / ls_s.rolling(63).std()) * np.sqrt(PORT_ANN_FACTOR)
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(np.arange(len(roll)), roll.values, lw=1.2, color='steelblue')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'63-Day Rolling Sharpe — Hybrid [{SECTOR_WEIGHTING}] Q={_q:.0%} LAM={_lam} MW={_use_mw}')
    ax.set_xlabel('Trading Day'); ax.set_ylabel('Rolling Sharpe')
    plt.tight_layout(); plt.show()

    # ── Sector Decomposition ─────────────────────────────────────────────────
    _sec_present = [s for s in ALL_SECTORS if f'l_{s}' in wf_port_df.columns or f's_{s}' in wf_port_df.columns]
    if _sec_present:
        import colorsys
        import matplotlib.colors as mcolors

        _cmap  = plt.cm.get_cmap('tab10', len(_sec_present))
        _scols = [_cmap(i % 10) for i in range(len(_sec_present))]
        _la_s  = wf_port_df['long_alloc'].values
        _sa_s  = wf_port_df['short_alloc'].values

        def _lighten(col, amount=0.45):
            h, l, s = colorsys.rgb_to_hls(*mcolors.to_rgb(col))
            return colorsys.hls_to_rgb(h, min(1.0, l + amount * (1.0 - l)), s)

        def _darken(col, amount=0.35):
            h, l, s = colorsys.rgb_to_hls(*mcolors.to_rgb(col))
            return colorsys.hls_to_rgb(h, max(0.0, l * (1.0 - amount)), s)

        def _make_sector_series(sec):
            lc = f'l_{sec}'
            sc = f's_{sec}'
            l_v = wf_port_df[lc].values if lc in wf_port_df.columns else np.full(len(wf_port_df), np.nan)
            s_v = wf_port_df[sc].values if sc in wf_port_df.columns else np.full(len(wf_port_df), np.nan)
            ret  = np.where(~np.isnan(l_v), l_v * _la_s,
                   np.where(~np.isnan(s_v), -s_v * _sa_s, 0.0))
            side = np.where(~np.isnan(l_v), 1,
                   np.where(~np.isnan(s_v), -1, 0))
            cum  = np.cumprod(1.0 + ret)
            return cum, side

        def _plot_segments(ax, x, cum, side, light_col, dark_col):
            n, i = len(x), 0
            while i < n:
                s = side[i]
                j = i
                while j < n and side[j] == s:
                    j += 1
                end = min(j, n - 1)
                sx, sy = x[i:end + 1], cum[i:end + 1]
                if s == 1:
                    ax.plot(sx, sy, color=light_col, lw=1.4, solid_capstyle='round')
                elif s == -1:
                    ax.plot(sx, sy, color=dark_col,  lw=1.4, ls='--', dash_capstyle='round')
                i = j

        _x = np.arange(len(wf_port_df))

        def _add_regime_bg(ax):
            _rdates = pd.DatetimeIndex(wf_port_df.index)
            _rreg   = fng_regime_series.reindex(_rdates).ffill()
            _rprev, _rstart = None, 0
            for _ri, _r in enumerate(_rreg):
                if _r != _rprev:
                    if _rprev is not None and str(_rprev) in REGIME_COLORS:
                        ax.axvspan(_rstart, _ri, alpha=0.18, color=REGIME_COLORS[str(_rprev)], lw=0, zorder=0)
                    _rprev, _rstart = _r, _ri
            if _rprev is not None and str(_rprev) in REGIME_COLORS:
                ax.axvspan(_rstart, len(_rreg)-1, alpha=0.18, color=REGIME_COLORS[str(_rprev)], lw=0, zorder=0)

        # Per-sector grid
        _ncols = 3
        _nrows = max(1, (len(_sec_present) + _ncols - 1) // _ncols)
        fig, axes = plt.subplots(_nrows, _ncols, figsize=(18, _nrows * 4))
        axes = np.array(axes).flatten()
        for _i, _sec in enumerate(_sec_present):
            _ax     = axes[_i]
            _base   = _scols[_i]
            _light  = _lighten(_base)
            _dark   = _darken(_base)
            _cum, _side = _make_sector_series(_sec)
            _add_regime_bg(_ax)
            _plot_segments(_ax, _x, _cum, _side, _light, _dark)
            _ax.set_xlim(_x[0], _x[-1])
            _ax.set_ylim(_cum.min() * 0.98, _cum.max() * 1.02)
            _ax.axhline(1, color='gray', lw=0.5, ls=':')
            _ax.set_title(_sec, fontsize=9)
            _l_v2 = wf_port_df[f'l_{_sec}'].values if f'l_{_sec}' in wf_port_df.columns else np.full(len(wf_port_df), np.nan)
            _s_v2 = wf_port_df[f's_{_sec}'].values if f's_{_sec}' in wf_port_df.columns else np.full(len(wf_port_df), np.nan)
            _comb = np.where(~np.isnan(_l_v2), _l_v2, np.where(~np.isnan(_s_v2), -_s_v2, np.nan))
            _comb_s = pd.Series(_comb).dropna()
            def _sh(r): m=r.mean()*252; v=r.std()*np.sqrt(252); return m/v if v>0 else np.nan
            _sh_val = _sh(_comb_s) if len(_comb_s)>=5 else np.nan
            if np.isfinite(_sh_val): _ax.text(0.03, 0.97, f'Sharpe: {_sh_val:+.2f}', transform=_ax.transAxes, fontsize=7, va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.75))
            _ax.legend(handles=[
                plt.Line2D([0],[0], color=_light, lw=1.4, label='Long (solid)'),
                plt.Line2D([0],[0], color=_dark,  lw=1.4, ls='--', label='Short (dashed)'),
            ], fontsize=7)
        for _j in range(len(_sec_present), len(axes)):
            axes[_j].set_visible(False)
        plt.suptitle(f'Per-Sector Equity Curves — Q={_q:.0%} LAM={_lam} MW={_use_mw}  [solid=long(light), dashed=short(dark)]', fontsize=10)
        plt.tight_layout(); plt.show()

        # Pure long / pure short decomposition per sector
        _sec_pure = [s for s in ALL_SECTORS if f'ul_{s}' in wf_port_df.columns]
        _nrows_p  = max(1, (len(_sec_pure) + _ncols - 1) // _ncols)
        def _make_pure_legs(sec):
            lc  = f'ul_{sec}'
            sc  = f'us_{sec}'
            l_v = wf_port_df[lc].values if lc in wf_port_df.columns else np.full(len(wf_port_df), np.nan)
            s_v = wf_port_df[sc].values if sc in wf_port_df.columns else np.full(len(wf_port_df), np.nan)
            l_ret = np.where(~np.isnan(l_v), l_v * _la_s, 0.0)
            s_ret = np.where(~np.isnan(s_v), -s_v * _sa_s, 0.0)
            return np.cumprod(1.0 + l_ret), np.cumprod(1.0 + s_ret)

        fig, axes = plt.subplots(_nrows_p, _ncols, figsize=(18, _nrows_p * 4))
        axes = np.array(axes).flatten()
        for _i, _sec in enumerate(_sec_pure):
            _ax    = axes[_i]
            _ci    = [s for s in ALL_SECTORS].index(_sec) if _sec in ALL_SECTORS else _i
            _base  = _scols[_ci % len(_scols)]
            _light = _lighten(_base)
            _dark  = _darken(_base)
            _lcum, _scum = _make_pure_legs(_sec)
            _add_regime_bg(_ax)
            _ax.plot(_x, _lcum, color=_light, lw=1.4)
            _ax.plot(_x, _scum, color=_dark,  lw=1.4, ls='--')
            _ymin = min(_lcum.min(), _scum.min()) * 0.98
            _ymax = max(_lcum.max(), _scum.max()) * 1.02
            _ax.set_xlim(_x[0], _x[-1])
            _ax.set_ylim(_ymin, _ymax)
            _ax.axhline(1, color='gray', lw=0.5, ls=':')
            _ax.set_title(_sec, fontsize=9)
            _l_s = pd.Series(wf_port_df[f'ul_{_sec}'].values if f'ul_{_sec}' in wf_port_df.columns else []).dropna()
            _s_s = pd.Series(wf_port_df[f'us_{_sec}'].values if f'us_{_sec}' in wf_port_df.columns else []).dropna()
            _l_sh = _sh(_l_s) if len(_l_s)>=5 else np.nan
            _s_sh = _sh(-_s_s) if len(_s_s)>=5 else np.nan
            _box = (f'L Sharpe: {_l_sh:+.2f}\n' if np.isfinite(_l_sh) else '') + (f'S Sharpe: {_s_sh:+.2f}' if np.isfinite(_s_sh) else '')
            if _box: _ax.text(0.03, 0.97, _box.strip(), transform=_ax.transAxes, fontsize=7, va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.75))
            _ax.legend(handles=[
                plt.Line2D([0],[0], color=_light, lw=1.4, label='Long (solid)'),
                plt.Line2D([0],[0], color=_dark,  lw=1.4, ls='--', label='Short (dashed)'),
            ], fontsize=7)
        for _j in range(len(_sec_pure), len(axes)):
            axes[_j].set_visible(False)
        plt.suptitle(f'Pure Long / Pure Short by Sector — Q={_q:.0%} LAM={_lam} MW={_use_mw}', fontsize=10)
        plt.tight_layout(); plt.show()

        # Aggregate
        fig, ax = plt.subplots(figsize=(14, 6))
        for _i, _sec in enumerate(_sec_present):
            _base  = _scols[_i]
            _light = _lighten(_base)
            _dark  = _darken(_base)
            _cum, _side = _make_sector_series(_sec)
            _plot_segments(ax, _x, _cum, _side, _light, _dark)
        ax.set_xlim(_x[0], _x[-1])
        ax.autoscale(axis='y')
        ax.axhline(1, color='gray', lw=0.7, ls=':')
        ax.set_title(f'All Sectors — Q={_q:.0%} LAM={_lam} MW={_use_mw}  [solid=long(light), dashed=short(dark)]', fontsize=10)
        ax.legend(handles=[plt.Line2D([0],[0], color=_scols[_i], lw=1.5, label=_sec_present[_i])
                            for _i in range(len(_sec_present))],
                  fontsize=7, ncol=4, loc='upper left')
        plt.tight_layout(); plt.show()

        